# Extension backend: private Colab CPU session

Load saved models and evidence from shared Drive. The extension can select
DistilBERT with BM25, BGE or fused evidence, or the corrected NLI reader.
The three DistilBERT options share classifier weights; their evidence differs.
No training, full-corpus retrieval, GPU, public tunnel or third-party web proxy is used.
This notebook contains its source and runs directly in the kernel. Requests arrive
from an authenticated CLI relay on your own computer, where Chrome still uses
`http://127.0.0.1:8000`. Backend and relay both bind only to their own loopback.

Use your paid Colab account with positive compute credits. This is a supervised
prototype on known benchmark claims, not arbitrary live claim verification or
an always-on service. Colab's normal resource and usage policies still apply.

**CLI route:** create a named CPU session, authorize Drive with `colab drivemount`,
then execute this notebook with `colab exec --timeout 600 -f ...`. See
`apps/COLAB_BACKEND.md` or `scripts/colab_extension.ps1` in the repository.
**Browser route:** select CPU/None as accelerator and run in order; use
`colab sessions` on your computer to identify this same runtime before relaying.

## 1. Restore the bundled source

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, subprocess, sys, time
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
SOURCES = json.loads('{"apps/backend/baseline_adapter.py": "\\"\\"\\"Adapter that runs the saved BM25-top-3 DistilBERT baseline on benchmark claims.\\n\\nReal inference, restricted to known AVeriTeC dev claims: the saved dev BM25\\nrankings supply each claim\'s three passages. Inputs are built exactly like\\nscripts/train_transformer_verifier.py::compose_text, and the model, label order\\nand token budgets come from the saved baseline run. Gold labels are never read.\\n\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\n\\nLABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\")\\nMAX_EVIDENCE = 3\\n\\n# Standing caveat. 0.52 accuracy / 0.347 macro-F1 are the saved baseline\'s\\n# results on the 500 official dev claims (final_dev_metrics.json).\\nBASELINE_NOTE = (\\"This is a simple baseline model. On the benchmark it agreed with the \\"\\n                 \\"reference label about half the time, so treat it as a rough lead only.\\")\\n\\n\\ndef compose_text(claim, passages, tokenizer, claim_budget, evidence_budget):\\n    \\"\\"\\"Mirror of scripts/train_transformer_verifier.py::compose_text. Keep in sync.\\"\\"\\"\\n    claim_tokens = tokenizer.encode(str(claim), add_special_tokens=False)[:claim_budget]\\n    sections = [f\\"Claim: {tokenizer.decode(claim_tokens, skip_special_tokens=True)}\\"]\\n    usable = [p for p in passages[:MAX_EVIDENCE] if str(p.get(\\"text\\") or \\"\\").strip()]\\n    if not usable:\\n        sections.append(\\"Evidence: No candidate passage was retrieved.\\")\\n    for i, p in enumerate(usable, start=1):\\n        tokens = tokenizer.encode(str(p[\\"text\\"]), add_special_tokens=False)[:evidence_budget]\\n        sections.append(f\\"Evidence {i}: {tokenizer.decode(tokens, skip_special_tokens=True)}\\")\\n    return \\"\\\\n\\".join(sections)\\n\\n\\ndef load_rankings(path):\\n    rows = {}\\n    with open(path, encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            if line.strip():\\n                row = json.loads(line)\\n                rows[str(row[\\"claim_id\\"])] = row\\n    return rows\\n\\n\\ndef lean_note(probabilities):\\n    \\"\\"\\"Plain-language strength of the model\'s lean. No percentages (not calibrated).\\"\\"\\"\\n    top = max(probabilities)\\n    if top >= 0.7:\\n        return \\"The model leaned clearly toward this label.\\"\\n    if top >= 0.45:\\n        return \\"The model leaned toward this label, but not strongly.\\"\\n    return \\"The model had no strong lean; the labels were close.\\"\\n\\n\\nclass BaselineAdapter:\\n    name = \\"baseline_distilbert_bm25_top3\\"\\n\\n    def __init__(self, rankings, predict_fn, *, model_id=\'distilbert_bm25\',\\n                 model_label=\'DistilBERT + BM25\', model_description=\'Keyword-selected evidence.\',\\n                 provenance=None, model_note=BASELINE_NOTE):\\n        \\"\\"\\"rankings: {claim_id: row with \'claim\' and \'retrieved\'}.\\n        predict_fn(claim, passages) -> four probabilities in LABELS order.\\"\\"\\"\\n        self.rankings = rankings\\n        self.predict_fn = predict_fn\\n        self._cache = {}\\n        self.model_id = model_id\\n        self.model_label = model_label\\n        self.model_description = model_description\\n        self.model_note = model_note\\n        if provenance:\\n            self.name = provenance\\n        self.explanation_text = \'A classifier read the claim and the three passages shown and chose this label. It does not give reasons.\'\\n\\n    def with_evidence(self, rankings_path, *, model_id, model_label, description, provenance):\\n        \\"\\"\\"Reuse classifier weights while keeping evidence and prediction caches separate.\\"\\"\\"\\n        return BaselineAdapter(load_rankings(rankings_path), self.predict_fn,\\n            model_id=model_id, model_label=model_label, model_description=description,\\n            provenance=provenance, model_note=\'This uses the BM25-trained classifier with a different evidence selection. Treat its output as preliminary.\')\\n\\n    @classmethod\\n    def from_paths(cls, baseline_dir, rankings_path, device=\\"cpu\\"):\\n        import torch\\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n\\n        baseline_dir = Path(baseline_dir)\\n        settings = json.loads((baseline_dir / \\"transformer_run_config.json\\").read_text(encoding=\\"utf-8\\"))\\n        model_dir = baseline_dir / \\"final_model\\"\\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=True)\\n        model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).to(device).eval()\\n        if [model.config.id2label.get(i) for i in range(4)] != list(LABELS):\\n            raise ValueError(\\"Saved model\'s label order differs from the canonical labels\\")\\n\\n        def predict(claim, passages):\\n            text = compose_text(claim, passages, tokenizer, settings[\\"claim_token_budget\\"],\\n                                settings[\\"evidence_token_budget_per_passage\\"])\\n            enc = tokenizer(text, truncation=True, max_length=settings[\\"max_length\\"], return_tensors=\\"pt\\")\\n            with torch.inference_mode():\\n                logits = model(**{k: v.to(device) for k, v in enc.items()}).logits.float()\\n            return logits.softmax(-1)[0].cpu().tolist()\\n\\n        return cls(load_rankings(rankings_path), predict)\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        q = (query or \\"\\").strip().lower()\\n        out = []\\n        for cid, row in self.rankings.items():\\n            if q in cid.lower() or q in str(row.get(\\"claim\\", \\"\\")).lower():\\n                out.append({\\"claim_id\\": cid, \\"claim\\": row.get(\\"claim\\", \\"\\")})\\n                if len(out) >= limit:\\n                    break\\n        return out\\n\\n    def get(self, claim_id):\\n        row = self.rankings.get(claim_id)\\n        if row is None:\\n            return None\\n        if claim_id not in self._cache:\\n            self._cache[claim_id] = self._verify(row)\\n        return self._cache[claim_id]\\n\\n    def _verify(self, row):\\n        claim = row[\\"claim\\"]\\n        passages = row.get(\\"retrieved\\") or []\\n        evidence = [{\\"passage_id\\": str(p[\\"passage_id\\"]), \\"text\\": p[\\"text\\"],\\n                     \\"url\\": p.get(\\"url\\") or \\"\\", \\"rank\\": i}\\n                    for i, p in enumerate(passages[:MAX_EVIDENCE], start=1)\\n                    if str(p.get(\\"text\\") or \\"\\").strip()]\\n        if not evidence:\\n            return {\\"claim_id\\": row[\\"claim_id\\"], \\"claim\\": claim, \\"status\\": \\"insufficient_sources\\",\\n                    \\"explanation\\": \\"No source passages were found for this claim.\\", \\"evidence\\": []}\\n        probs = self.predict_fn(claim, passages)\\n        verdict = LABELS[max(range(4), key=lambda i: probs[i])]\\n        return {\\"claim_id\\": row[\\"claim_id\\"], \\"claim\\": claim, \\"status\\": \\"completed\\", \\"verdict\\": verdict,\\n                \\"explanation\\": self.explanation_text,\\n                \\"uncertainty_note\\": lean_note(probs) + \\" \\" + self.model_note,\\n                \\"evidence\\": evidence}\\n", "apps/backend/colab_runtime.py": "\\"\\"\\"Start the private benchmark backend inside a Colab CPU kernel.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport importlib.util\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport shutil\\nimport sys\\nimport threading\\nimport time\\nfrom http.server import ThreadingHTTPServer\\n\\nRUN_PATH = Path(\\"outputs/initial_ml_solution_rerun_bs16/transformer/bm25_top3\\")\\nRANKINGS_PATH = Path(\\"outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl\\")\\nBGE_PATH = Path(\'outputs/initial_ml_solution_rerun_bs16/dense/dev_bge_small_v15/dev_rankings.jsonl\')\\nFUSION_PATH = Path(\'outputs/initial_ml_solution_rerun_bs16/hybrid/dev_rrf60/hybrid_rankings.jsonl\')\\nNLI_RUN_PATH = Path(\'outputs/evidence_reader_followup_v2/pair/raw_sample_d01\')\\nNLI_PREP_PATH = Path(\'outputs/evidence_aware_reader_v1/prepared\')\\n_ACTIVE = None\\n\\n\\ndef file_hash(path):\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef stage_artifacts(data_root, cache_root):\\n    \\"\\"\\"Copy a small serving bundle atomically; never write to shared inputs.\\"\\"\\"\\n    data_root, cache_root = Path(data_root).resolve(), Path(cache_root).resolve()\\n    run = data_root / RUN_PATH\\n    model = run / \\"final_model\\"\\n    required = [run / \\"transformer_run_config.json\\", model / \\"config.json\\", data_root / RANKINGS_PATH]\\n    missing = [str(path) for path in required if not path.is_file()]\\n    if not model.is_dir() or not any((model / name).is_file() for name in (\\"model.safetensors\\", \\"pytorch_model.bin\\")):\\n        missing.append(str(model / \\"model.safetensors\\"))\\n    if missing:\\n        raise FileNotFoundError(\\"Check the mounted shared Drive and DATA_ROOT:\\\\n\\" + \\"\\\\n\\".join(missing))\\n    if cache_root == data_root or cache_root.is_relative_to(data_root):\\n        raise ValueError(\\"Use a runtime cache outside the shared input folder\\")\\n    # final_model contains the saved tokenizer and weights. Training checkpoint\\n    # subdirectories and the full evidence corpus are deliberately not copied.\\n    sources = [(path, Path(\'baseline/final_model\') / path.name) for path in sorted(model.iterdir()) if path.is_file()]\\n    sources += [(run / \'transformer_run_config.json\', Path(\'baseline/transformer_run_config.json\')),\\n                (data_root / RANKINGS_PATH, Path(\'dev_rankings.jsonl\'))]\\n    available = {\'distilbert_bm25\': True}\\n    for model_id, source, name in ((\'distilbert_bge\', data_root / BGE_PATH, \'bge_rankings.jsonl\'),\\n                                    (\'distilbert_fusion\', data_root / FUSION_PATH, \'fusion_rankings.jsonl\')):\\n        available[model_id] = source.is_file()\\n        if source.is_file():\\n            sources.append((source, Path(name)))\\n    nli_run, nli_prep = data_root / NLI_RUN_PATH, data_root / NLI_PREP_PATH\\n    nli_required = [nli_run / \'reader_run.json\', nli_run / \'final_model/config.json\',\\n                    nli_run / \'final_model/model.safetensors\', nli_prep / \'raw/dev_rankings.jsonl\', nli_prep / \'preparation.json\']\\n    available[\'nli_reader\'] = all(path.is_file() for path in nli_required)\\n    if available[\'nli_reader\']:\\n        sources += [(path, Path(\'nli_reader/final_model\') / path.name) for path in sorted((nli_run / \'final_model\').iterdir()) if path.is_file()]\\n        sources += [(nli_run / \'reader_run.json\', Path(\'nli_reader/reader_run.json\')),\\n                    (nli_prep / \'raw/dev_rankings.jsonl\', Path(\'nli_rankings.jsonl\')),\\n                    (nli_prep / \'preparation.json\', Path(\'nli_preparation.json\'))]\\n    manifest_path = cache_root / \\"serving_manifest.json\\"\\n    previous = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}\\n    previous = previous if previous.get(\\"data_root\\") == str(data_root) else {}\\n    manifest = {\\"data_root\\": str(data_root), \\"files\\": {}, \'available_pipelines\': available}\\n    for source, relative in sources:\\n        target = cache_root / relative\\n        stat = source.stat()\\n        fingerprint = {\\"source\\": str(source), \\"size\\": stat.st_size, \\"mtime_ns\\": stat.st_mtime_ns}\\n        old = previous.get(\\"files\\", {}).get(relative.as_posix(), {})\\n        unchanged = all(old.get(key) == value for key, value in fingerprint.items())\\n        if unchanged and target.is_file() and file_hash(target) == old.get(\\"sha256\\"):\\n            print(\\"Reusing\\", relative, flush=True)\\n            manifest[\\"files\\"][relative.as_posix()] = old\\n            continue\\n        target.parent.mkdir(parents=True, exist_ok=True)\\n        temporary = target.with_name(target.name + \\".copying\\")\\n        print(f\\"Copying {relative} ({stat.st_size / 1e6:.1f} MB)\\", flush=True)\\n        shutil.copyfile(source, temporary)\\n        after = source.stat()\\n        if (after.st_size, after.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):\\n            raise RuntimeError(\\"Source changed during copy; rerun after it is stable: \\" + str(source))\\n        digest = file_hash(temporary)\\n        os.replace(temporary, target)\\n        manifest[\\"files\\"][relative.as_posix()] = {**fingerprint, \\"sha256\\": digest}\\n    cache_root.mkdir(parents=True, exist_ok=True)\\n    temporary = manifest_path.with_suffix(\\".json.tmp\\")\\n    temporary.write_text(json.dumps(manifest, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    os.replace(temporary, manifest_path)\\n    return cache_root / \\"baseline\\", cache_root / \\"dev_rankings.jsonl\\", manifest\\n\\n\\ndef _module(name, filename):\\n    spec = importlib.util.spec_from_file_location(name, Path(__file__).with_name(filename))\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    return module\\n\\n\\nclass SerializedAdapter:\\n    \\"\\"\\"Keep concurrent requests from oversubscribing a small CPU runtime.\\"\\"\\"\\n    def __init__(self, adapter, lock=None):\\n        self.adapter, self.name = adapter, adapter.name\\n        self.lock = lock or threading.Lock()\\n\\n    def __getattr__(self, name):\\n        return getattr(self.adapter, name)\\n\\n    def get(self, claim_id):\\n        with self.lock:\\n            return self.adapter.get(claim_id)\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        return self.adapter.list_claims(query, limit)\\n\\n    def select_model(self, model_id=None):\\n        selector = getattr(self.adapter, \'select_model\', None)\\n        if selector:\\n            return SerializedAdapter(selector(model_id), self.lock)\\n        if model_id is not None and model_id != getattr(self.adapter, \'model_id\', self.name):\\n            raise ValueError(\'Unknown model_id\')\\n        return self\\n\\n\\ndef start_backend(data_root, cache_root=\\"/content/claim_check_serving\\", *, port=8000,\\n                  cpu_threads=2, adapter_factory=None):\\n    \\"\\"\\"Load once, serve on runtime loopback, and return to the notebook kernel.\\n\\n    The local authenticated CLI relay is the only intended client. No public\\n    tunnel, retraining, new retrieval, or artificial runtime keepalive is used.\\n    \\"\\"\\"\\n    global _ACTIVE\\n    settings = (str(Path(data_root).resolve()), str(Path(cache_root).resolve()), port, cpu_threads)\\n    if _ACTIVE:\\n        if _ACTIVE[\\"settings\\"] != settings:\\n            raise RuntimeError(\\"Stop the existing backend before changing its settings\\")\\n        if _ACTIVE[\\"thread\\"].is_alive():\\n            return {**_ACTIVE[\\"info\\"], \\"reused\\": True}\\n        raise RuntimeError(\\"Previous backend stopped unexpectedly; call stop_backend() and restart\\")\\n    if not 0 <= port <= 65535 or not 1 <= cpu_threads <= 64:\\n        raise ValueError(\\"Invalid port or CPU thread count\\")\\n    os.environ[\\"USE_TF\\"] = \\"0\\"\\n    os.environ[\\"USE_FLAX\\"] = \\"0\\"\\n    os.environ[\\"TOKENIZERS_PARALLELISM\\"] = \\"false\\"\\n    baseline, rankings, manifest = stage_artifacts(data_root, cache_root)\\n    if adapter_factory is None:\\n        import torch\\n        torch.set_num_threads(cpu_threads)\\n        directory = str(Path(__file__).parent)\\n        if directory not in sys.path:\\n            sys.path.insert(0, directory)\\n    started = time.monotonic()\\n    print(\\"Loading the saved model and rankings on CPU...\\", flush=True)\\n    if adapter_factory is None:\\n        adapter = _module(\'_claim_check_registry\', \'model_registry.py\').load_registry(cache_root, manifest, device=\'cpu\')\\n    else:\\n        adapter = adapter_factory(baseline, rankings, \'cpu\')\\n    adapter = SerializedAdapter(adapter)\\n    server_module = _module(\\"_claim_check_http\\", \\"server.py\\")\\n    server = ThreadingHTTPServer((\\"127.0.0.1\\", port), server_module.make_handler(adapter))\\n    thread = threading.Thread(target=server.serve_forever, name=\\"claim-check-backend\\", daemon=True)\\n    thread.start()\\n    info = {\\"url\\": f\\"http://127.0.0.1:{server.server_port}\\", \\"adapter\\": adapter.name,\\n            \\"device\\": \\"cpu\\", \\"loaded_in_seconds\\": round(time.monotonic() - started, 2),\\n            \\"claim_count\\": len(adapter.list_claims(\\"\\", 10000)), \\"cache_root\\": str(Path(cache_root)),\\n            \\"manifest\\": str(Path(cache_root) / \\"serving_manifest.json\\"), \\"reused\\": False}\\n    if callable(getattr(adapter, \'list_models\', None)):\\n        info[\'models\'] = adapter.list_models()\\n    _ACTIVE = {\\"server\\": server, \\"thread\\": thread, \\"adapter\\": adapter,\\n               \\"settings\\": settings, \\"info\\": info}\\n    print(\\"Private backend ready:\\", info[\\"url\\"], flush=True)\\n    return info\\n\\n\\ndef stop_backend():\\n    global _ACTIVE\\n    if _ACTIVE:\\n        _ACTIVE[\\"server\\"].shutdown()\\n        _ACTIVE[\\"server\\"].server_close()\\n        _ACTIVE[\\"thread\\"].join(timeout=5)\\n        _ACTIVE = None\\n        print(\\"Backend stopped. Stop the Colab session separately when finished.\\", flush=True)\\n", "apps/backend/model_registry.py": "\\"\\"\\"Explicit serving choices; unknown or unavailable choices never silently fall back.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\n\\nfrom baseline_adapter import BaselineAdapter\\n\\nPROFILES = (\\n    {\'id\': \'distilbert_bm25\', \'label\': \'DistilBERT + BM25\', \'description\': \'Original classifier with keyword-selected evidence.\'},\\n    {\'id\': \'distilbert_bge\', \'label\': \'DistilBERT + BGE\', \'description\': \'The same classifier with semantic-retrieval evidence.\'},\\n    {\'id\': \'distilbert_fusion\', \'label\': \'DistilBERT + BM25/BGE fusion\', \'description\': \'The same classifier with fused keyword and semantic evidence.\'},\\n    {\'id\': \'nli_reader\', \'label\': \'Corrected NLI reader\', \'description\': \'A trained reader compares each BM25 passage with the claim.\'},\\n)\\n\\n\\nclass ModelRegistry:\\n    name = \'model_registry\'\\n    model_selection = True\\n    default_model_id = \'distilbert_bm25\'\\n\\n    def __init__(self, adapters, unavailable=None):\\n        self.adapters = dict(adapters)\\n        self.unavailable = dict(unavailable or {})\\n        if self.default_model_id not in self.adapters:\\n            raise ValueError(\'The original baseline must be available\')\\n        baseline = self.adapters[self.default_model_id]\\n        for adapter in self.adapters.values():\\n            if set(adapter.rankings) != set(baseline.rankings):\\n                raise ValueError(\'Serving choices must cover the same claim IDs\')\\n            if any(str(row[\'claim\']) != str(baseline.rankings[cid][\'claim\']) for cid, row in adapter.rankings.items()):\\n                raise ValueError(\'Serving choices disagree on benchmark claim text\')\\n\\n    def list_models(self):\\n        return [{**profile, \'available\': profile[\'id\'] in self.adapters,\\n                 \'unavailable_reason\': self.unavailable.get(profile[\'id\'])} for profile in PROFILES]\\n\\n    def select_model(self, model_id=None):\\n        model_id = self.default_model_id if model_id is None else model_id\\n        if not isinstance(model_id, str) or model_id not in {profile[\'id\'] for profile in PROFILES}:\\n            raise ValueError(\'Unknown model_id\')\\n        if model_id not in self.adapters:\\n            raise LookupError(self.unavailable.get(model_id, \'This model is not loaded\'))\\n        return self.adapters[model_id]\\n\\n    def get(self, claim_id):\\n        return self.select_model().get(claim_id)\\n\\n    def list_claims(self, query=\'\', limit=50):\\n        return self.select_model().list_claims(query, limit)\\n\\n\\ndef load_registry(cache_root, manifest, *, device=\'cpu\'):\\n    cache_root = Path(cache_root)\\n    baseline = BaselineAdapter.from_paths(cache_root / \'baseline\', cache_root / \'dev_rankings.jsonl\', device)\\n    adapters, unavailable = {\'distilbert_bm25\': baseline}, {}\\n    available = manifest.get(\'available_pipelines\', {\'distilbert_bm25\': True})\\n    for model_id, filename, provenance in (\\n        (\'distilbert_bge\', \'bge_rankings.jsonl\', \'distilbert_bge_top3\'),\\n        (\'distilbert_fusion\', \'fusion_rankings.jsonl\', \'distilbert_fusion_top3\'),\\n    ):\\n        if available.get(model_id):\\n            profile = next(item for item in PROFILES if item[\'id\'] == model_id)\\n            adapters[model_id] = baseline.with_evidence(cache_root / filename, model_id=model_id,\\n                model_label=profile[\'label\'], description=profile[\'description\'], provenance=provenance)\\n        else:\\n            unavailable[model_id] = \'Its saved evidence file was not found in shared Drive.\'\\n    if available.get(\'nli_reader\'):\\n        try:\\n            from nli_adapter import NliReaderAdapter\\n            print(\'Loading the corrected NLI reader on CPU...\', flush=True)\\n            adapters[\'nli_reader\'] = NliReaderAdapter.from_paths(cache_root / \'nli_reader\',\\n                cache_root / \'nli_rankings.jsonl\', cache_root / \'nli_preparation.json\', device)\\n        except Exception as error:\\n            print(\'NLI reader unavailable:\', type(error).__name__, str(error), flush=True)\\n            unavailable[\'nli_reader\'] = \'The reader could not be loaded; check the Colab notebook output.\'\\n    else:\\n        unavailable[\'nli_reader\'] = \'Its checkpoint or prepared evidence was not found in shared Drive.\'\\n    return ModelRegistry(adapters, unavailable)\\n", "apps/backend/nli_adapter.py": "\\"\\"\\"Serve the corrected saved pair reader using its original prediction-time inputs.\\"\\"\\"\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom baseline_adapter import BaselineAdapter, LABELS, load_rankings\\n\\n\\nclass NliReaderAdapter(BaselineAdapter):\\n    @classmethod\\n    def from_paths(cls, run_dir, rankings_path, preparation_path, device=\'cpu\'):\\n        import torch\\n        from transformers import AutoTokenizer\\n        from src.verification.pair_reader import EvidencePairReader, PairDataset, PairCollator\\n        from src.retrieval.evidence_quality import QUALITY_NAMES\\n\\n        run_dir, rankings_path, preparation_path = map(Path, (run_dir, rankings_path, preparation_path))\\n        run = json.loads((run_dir / \'reader_run.json\').read_text())\\n        preparation = json.loads(preparation_path.read_text())\\n        if hashlib.sha256(preparation_path.read_bytes()).hexdigest() != run[\'preparation_sha256\']:\\n            raise ValueError(\'Reader preparation differs from the saved training run\')\\n        if hashlib.sha256(rankings_path.read_bytes()).hexdigest() != run[\'input_hashes\'][\'dev_rankings\']:\\n            raise ValueError(\'Reader evidence differs from the saved evaluation inputs\')\\n        rankings = load_rankings(rankings_path)\\n        for row in rankings.values():\\n            for item in (row.get(\'retrieved\') or [])[:3]:\\n                if len(item.get(\'quality\', [])) != len(QUALITY_NAMES):\\n                    raise ValueError(\'Reader evidence lacks its prediction-time quality features\')\\n        model_dir = run_dir / \'final_model\'\\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=True)\\n        model = EvidencePairReader.from_pretrained(model_dir, local_files_only=True).to(device).eval()\\n        if [model.config.id2label.get(i) for i in range(4)] != list(LABELS):\\n            raise ValueError(\'Reader label order differs from the canonical labels\')\\n        if model.config.loss_normalization != \'sample_mean\':\\n            raise ValueError(\'Expected the corrected reader checkpoint\')\\n        collator = PairCollator(tokenizer)\\n\\n        def predict(claim, passages):\\n            # No gold label is loaded or supplied, even as a placeholder.\\n            dataset = PairDataset([{\'claim\': claim, \'retrieved_evidence\': passages}], tokenizer,\\n                max_length=preparation[\'max_length\'], claim_budget=preparation[\'claim_budget\'], show_progress=False)\\n            batch = collator([dataset[0]])\\n            if \'labels\' in batch:\\n                raise ValueError(\'Serving inputs must be unlabeled\')\\n            with torch.inference_mode():\\n                return model(**{key: value.to(device) for key, value in batch.items()}).logits.float().softmax(-1)[0].cpu().tolist()\\n\\n        adapter = cls(rankings, predict, model_id=\'nli_reader\', model_label=\'Corrected NLI reader\',\\n            model_description=\'A trained reader checks each BM25 passage against the claim.\',\\n            provenance=\'corrected_nli_reader_bm25_top3\',\\n            model_note=\'This trained evidence reader is experimental. Read the sources before deciding.\')\\n        adapter.explanation_text = \'A trained reader compared the claim with each displayed passage and combined them into this verdict. It does not generate reasons.\'\\n        return adapter\\n", "apps/backend/requirements-model.txt": "# Only needed for --baseline-dir. PyTorch must also be installed.\\ntransformers==4.57.1\\nsafetensors\\nsentencepiece>=0.2,<0.3\\nscikit-learn>=1.5,<2\\n", "apps/backend/server.py": "\\"\\"\\"Development HTTP boundary for the screening assistant (stdlib only).\\n\\nRoutes: GET /health, GET /benchmark/claims, POST /verify.\\nSee docs/INTEGRATION_CONTRACT.md. Supports fixture replay and the saved baseline\\nadapter; other model adapters plug in through the same boundary.\\n\\"\\"\\"\\nimport argparse\\nimport json\\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\nfrom pathlib import Path\\nfrom urllib.parse import parse_qs, urlparse\\n\\nLABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\")\\nMAX_EVIDENCE = 3\\nMAX_CLAIM_CHARS = 2000\\nMAX_BODY_BYTES = 64 * 1024\\nFIXTURE_PATH = Path(__file__).parent / \\"fixtures\\" / \\"demo_claims.json\\"\\n\\n\\nclass Adapter:\\n    \\"\\"\\"Verifier boundary: ordered passages in, verdict fields out.\\"\\"\\"\\n\\n    name = \\"base\\"\\n\\n    def verify(self, claim, passages):  # pragma: no cover - interface\\n        raise NotImplementedError\\n\\n\\nclass FixtureReplayAdapter(Adapter):\\n    \\"\\"\\"Replays saved fixture results. This is NOT new model inference.\\"\\"\\"\\n\\n    name = \\"fixture_replay\\"\\n\\n    def __init__(self, path=FIXTURE_PATH):\\n        data = json.loads(Path(path).read_text(encoding=\\"utf-8\\"))\\n        self.records = {r[\\"claim_id\\"]: r for r in data[\\"claims\\"]}\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        q = (query or \\"\\").strip().lower()\\n        rows = [{\\"claim_id\\": k, \\"claim\\": v[\\"claim\\"]} for k, v in self.records.items()\\n                if q in k.lower() or q in v[\\"claim\\"].lower()]\\n        return rows[:limit]\\n\\n    def get(self, claim_id):\\n        return self.records.get(claim_id)\\n\\n\\ndef normalize_evidence(items):\\n    \\"\\"\\"Order by rank, keep at most three, renumber ranks 1..n.\\"\\"\\"\\n    ordered = sorted(items, key=lambda e: e.get(\\"rank\\", 0))[:MAX_EVIDENCE]\\n    return [\\n        {\\"passage_id\\": e[\\"passage_id\\"], \\"text\\": e[\\"text\\"], \\"url\\": e[\\"url\\"], \\"rank\\": i}\\n        for i, e in enumerate(ordered, start=1)\\n    ]\\n\\n\\ndef error_response(mode, message):\\n    return {\\"mode\\": mode, \\"status\\": \\"error\\", \\"claim\\": None, \\"verdict\\": None,\\n            \\"explanation\\": message, \\"uncertainty_note\\": None, \\"evidence\\": []}\\n\\n\\ndef model_catalog(adapter):\\n    if callable(getattr(adapter, \'list_models\', None)):\\n        models = adapter.list_models()\\n    else:\\n        models = [{\'id\': getattr(adapter, \'model_id\', adapter.name),\\n                   \'label\': getattr(adapter, \'model_label\', \'Synthetic demo fixtures\' if adapter.name == \'fixture_replay\' else adapter.name),\\n                   \'description\': getattr(adapter, \'model_description\', \'\'), \'available\': True}]\\n    return {\'default_model_id\': getattr(adapter, \'default_model_id\', models[0][\'id\']), \'models\': models}\\n\\n\\ndef select_adapter(adapter, model_id):\\n    if callable(getattr(adapter, \'select_model\', None)):\\n        return adapter.select_model(model_id)\\n    expected = getattr(adapter, \'model_id\', adapter.name)\\n    if model_id is not None and model_id != expected:\\n        raise ValueError(\'Unknown model_id\')\\n    return adapter\\n\\n\\ndef handle_verify(payload, adapter):\\n    \\"\\"\\"Return (http_status, response_dict). Pure function so tests need no socket.\\"\\"\\"\\n    if not isinstance(payload, dict):\\n        return 400, error_response(None, \\"Request body must be a JSON object.\\")\\n    mode = payload.get(\\"mode\\")\\n    if mode == \\"live\\":\\n        claim = payload.get(\\"claim\\")\\n        if not isinstance(claim, str) or not claim.strip():\\n            return 400, error_response(mode, \\"A non-empty claim is required.\\")\\n        if len(claim) > MAX_CLAIM_CHARS:\\n            return 400, error_response(mode, f\\"Claim exceeds {MAX_CLAIM_CHARS} characters.\\")\\n        # No retrieval path exists for arbitrary claims. Never attach another\\n        # claim\'s evidence; report the limitation instead.\\n        return 200, {\\"mode\\": \\"live\\", \\"status\\": \\"insufficient_sources\\", \\"claim\\": claim.strip(),\\n                     \\"verdict\\": None,\\n                     \\"explanation\\": \\"Live checking is not available yet: no source retrieval is set up for new claims.\\",\\n                     \\"uncertainty_note\\": \\"Nothing was checked. Do not treat this as a result.\\",\\n                     \\"evidence\\": []}\\n    if mode == \\"benchmark\\":\\n        try:\\n            selected = select_adapter(adapter, payload.get(\'model_id\'))\\n        except ValueError:\\n            return 400, error_response(mode, \'Unknown model selection. Choose an available model.\')\\n        except LookupError as error:\\n            return 503, error_response(mode, str(error))\\n        claim_id = payload.get(\\"claim_id\\")\\n        record = selected.get(claim_id) if isinstance(claim_id, str) else None\\n        if record is None:\\n            return 404, error_response(mode, \\"Unknown benchmark claim_id.\\")\\n        if record.get(\\"simulate\\") == \\"error\\":\\n            return 503, error_response(mode, \\"Simulated backend failure (fixture).\\")\\n        status = record.get(\\"status\\", \\"completed\\")\\n        verdict = record.get(\\"verdict\\") if status == \\"completed\\" else None\\n        if status == \\"completed\\" and verdict not in LABELS:\\n            return 500, error_response(mode, \\"Fixture has an invalid verdict label.\\")\\n        return 200, {\\"mode\\": \\"benchmark\\", \\"status\\": status, \\"claim\\": record[\\"claim\\"],\\n                     \\"verdict\\": verdict, \\"explanation\\": record.get(\\"explanation\\"),\\n                     \\"uncertainty_note\\": record.get(\\"uncertainty_note\\"),\\n                     \\"evidence\\": normalize_evidence(record.get(\\"evidence\\", [])),\\n                     \\"provenance\\": selected.name,\\n                     \'model_id\': getattr(selected, \'model_id\', selected.name),\\n                     \'model_label\': getattr(selected, \'model_label\', \'Synthetic demo fixtures\' if selected.name == \'fixture_replay\' else selected.name)}\\n    return 400, error_response(mode if isinstance(mode, str) else None,\\n                               \\"mode must be \'benchmark\' or \'live\'.\\")\\n\\n\\ndef make_handler(adapter):\\n    class Handler(BaseHTTPRequestHandler):\\n        def _send(self, code, body):\\n            data = json.dumps(body, ensure_ascii=False).encode(\\"utf-8\\")\\n            self.send_response(code)\\n            self.send_header(\\"Content-Type\\", \\"application/json; charset=utf-8\\")\\n            self.send_header(\\"Content-Length\\", str(len(data)))\\n            self.send_header(\\"Access-Control-Allow-Origin\\", \\"*\\")\\n            self.send_header(\\"Access-Control-Allow-Methods\\", \\"GET, POST, OPTIONS\\")\\n            self.send_header(\\"Access-Control-Allow-Headers\\", \\"Content-Type\\")\\n            self.end_headers()\\n            self.wfile.write(data)\\n\\n        def do_OPTIONS(self):\\n            self._send(204, {})\\n\\n        def do_GET(self):\\n            if self.path == \\"/health\\":\\n                self._send(200, {\\"ok\\": True, \\"adapter\\": adapter.name,\\n                    \'model_selection\': bool(getattr(adapter, \'model_selection\', False))})\\n            elif self.path == \'/models\':\\n                self._send(200, model_catalog(adapter))\\n            elif urlparse(self.path).path == \\"/benchmark/claims\\":\\n                params = parse_qs(urlparse(self.path).query)\\n                query = (params.get(\\"q\\") or [\\"\\"])[0][:200]\\n                try:\\n                    limit = max(1, min(1000, int((params.get(\\"limit\\") or [\\"50\\"])[0])))\\n                except ValueError:\\n                    limit = 50\\n                self._send(200, {\\"claims\\": adapter.list_claims(query, limit), \\"provenance\\": adapter.name})\\n            else:\\n                self._send(404, error_response(None, \\"Not found.\\"))\\n\\n        def do_POST(self):\\n            if self.path != \\"/verify\\":\\n                return self._send(404, error_response(None, \\"Not found.\\"))\\n            try:\\n                length = int(self.headers.get(\\"Content-Length\\", \\"0\\"))\\n            except ValueError:\\n                length = -1\\n            if length < 0 or length > MAX_BODY_BYTES:\\n                return self._send(413, error_response(None, \\"Body missing or too large.\\"))\\n            try:\\n                payload = json.loads(self.rfile.read(length).decode(\\"utf-8\\"))\\n            except (UnicodeDecodeError, json.JSONDecodeError):\\n                return self._send(400, error_response(None, \\"Invalid JSON.\\"))\\n            code, body = handle_verify(payload, adapter)\\n            self._send(code, body)\\n\\n        def log_message(self, fmt, *args):  # keep claim text out of logs\\n            pass\\n\\n    return Handler\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--host\\", default=\\"127.0.0.1\\")\\n    parser.add_argument(\\"--port\\", type=int, default=8000)\\n    parser.add_argument(\\"--baseline-dir\\", help=\\"Saved baseline run folder (final_model/, transformer_run_config.json)\\")\\n    parser.add_argument(\\"--dev-rankings\\", help=\\"Saved dev_rankings.jsonl (BM25 top 20)\\")\\n    parser.add_argument(\\"--device\\", default=\\"cpu\\")\\n    parser.add_argument(\'--serving-cache\', type=Path, help=\'Staged cache with serving_manifest.json and selectable models\')\\n    args = parser.parse_args()\\n    if args.serving_cache:\\n        if args.baseline_dir or args.dev_rankings:\\n            parser.error(\'--serving-cache cannot be combined with single-baseline paths\')\\n        from model_registry import load_registry\\n        manifest = json.loads((args.serving_cache / \'serving_manifest.json\').read_text())\\n        adapter = load_registry(args.serving_cache, manifest, device=args.device)\\n    elif args.baseline_dir or args.dev_rankings:\\n        if not (args.baseline_dir and args.dev_rankings):\\n            parser.error(\\"--baseline-dir and --dev-rankings must be given together\\")\\n        from baseline_adapter import BaselineAdapter\\n        adapter = BaselineAdapter.from_paths(args.baseline_dir, args.dev_rankings, args.device)\\n    else:\\n        adapter = FixtureReplayAdapter()\\n    server = ThreadingHTTPServer((args.host, args.port), make_handler(adapter))\\n    print(f\\"Backend on http://{args.host}:{args.port} (adapter: {adapter.name})\\")\\n    server.serve_forever()\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "apps/browser-extension/manifest.json": "{\\n  \\"manifest_version\\": 3,\\n  \\"name\\": \\"Claim Check (course prototype)\\",\\n  \\"version\\": \\"0.2.0\\",\\n  \\"description\\": \\"Select a claim, request a check, and review preliminary verdict and sources. Prototype for CMU 10-718.\\",\\n  \\"action\\": { \\"default_title\\": \\"Check a claim\\", \\"default_popup\\": \\"popup.html\\" },\\n  \\"permissions\\": [\\"activeTab\\", \\"scripting\\", \\"clipboardRead\\", \\"storage\\"],\\n  \\"host_permissions\\": [\\"http://127.0.0.1:8000/*\\", \\"http://localhost:8000/*\\"]\\n}\\n", "apps/browser-extension/mock.js": "// Built-in SYNTHETIC samples so the UI can be exercised without a backend.\\n// These are not real checks and not model output. Same shape as the contract.\\nconst MOCK_SCENARIOS = {\\n  supported: { label: \\"Supported (2 sources)\\", status: \\"completed\\", verdict: \\"Supported\\",\\n    explanation: \\"The sample passages agree with the claim.\\", uncertainty_note: \\"Sample data only.\\",\\n    evidence: [\\n      { passage_id: \\"mock-1\\", rank: 1, url: \\"https://example.com/a\\", text: \\"Sample passage that agrees with the claim.\\" },\\n      { passage_id: \\"mock-2\\", rank: 2, url: \\"https://example.com/b\\", text: \\"A second sample passage with the same conclusion.\\" }] },\\n  refuted: { label: \\"Refuted (3 sources)\\", status: \\"completed\\", verdict: \\"Refuted\\",\\n    explanation: \\"The sample passages contradict the claim.\\", uncertainty_note: \\"Sample data only.\\",\\n    evidence: [1, 2, 3].map(i => ({ passage_id: \\"mock-r\\" + i, rank: i, url: \\"https://example.com/r\\" + i,\\n      text: \\"Sample passage \\" + i + \\" that contradicts the claim.\\" })) },\\n  nei: { label: \\"Not Enough Evidence\\", status: \\"completed\\", verdict: \\"Not Enough Evidence\\",\\n    explanation: \\"The sample passages do not settle the claim.\\", uncertainty_note: \\"Additional sources would be needed.\\",\\n    evidence: [{ passage_id: \\"mock-n1\\", rank: 1, url: \\"https://example.com/n\\", text: \\"Related but inconclusive sample passage.\\" }] },\\n  conflicting: { label: \\"Conflicting Evidence\\", status: \\"completed\\", verdict: \\"Conflicting Evidence\\",\\n    explanation: \\"One sample passage supports the claim and another contradicts it.\\", uncertainty_note: \\"The sources disagree; read both.\\",\\n    evidence: [\\n      { passage_id: \\"mock-c1\\", rank: 1, url: \\"https://example.com/c1\\", text: \\"Sample passage in favour.\\" },\\n      { passage_id: \\"mock-c2\\", rank: 2, url: \\"https://example.com/c2\\", text: \\"Sample passage against.\\" }] },\\n  insufficient: { label: \\"No sources found\\", status: \\"insufficient_sources\\", verdict: null,\\n    explanation: \\"No usable source passages were found for this claim.\\", uncertainty_note: null, evidence: [] },\\n  error: { label: \\"Service error\\", status: \\"error\\", verdict: null,\\n    explanation: \\"The service could not complete the check.\\", uncertainty_note: null, evidence: [] },\\n};\\n\\nasync function mockVerify(claim, scenarioKey) {\\n  await new Promise(r => setTimeout(r, 700)); // visible loading state\\n  return Object.assign({ mode: \\"mock\\", claim: claim }, MOCK_SCENARIOS[scenarioKey]);\\n}\\n", "apps/browser-extension/popup.css": ":root {\\n  --bg: #ffffff; --fg: #1d2330; --muted: #5b6475; --line: #e1e4ea; --card: #f6f7f9;\\n  --accent: #2f5bd8; --accent-fg: #ffffff;\\n  --supported: #17713a; --refuted: #a8271f; --nei: #5b6475; --conflict: #8a5a00; --warn: #8a5a00;\\n}\\n@media (prefers-color-scheme: dark) {\\n  :root { --bg: #171a21; --fg: #e8eaf0; --muted: #a3abbb; --line: #333a49; --card: #20242e;\\n    --accent: #7d9bff; --accent-fg: #101319;\\n    --supported: #5fd08a; --refuted: #ff8f86; --nei: #a3abbb; --conflict: #f0b84d; --warn: #f0b84d; }\\n}\\n* { box-sizing: border-box; }\\nbody { width: 400px; margin: 0; padding: 14px 14px 12px; background: var(--bg); color: var(--fg);\\n  font: 13px/1.45 system-ui, -apple-system, \\"Segoe UI\\", sans-serif; }\\n\\nheader { margin-bottom: 10px; }\\nh1 { display: flex; align-items: center; gap: 8px; font-size: 16px; margin: 0; letter-spacing: -.01em; }\\n.logo { display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 7px;\\n  background: var(--accent); color: var(--accent-fg); font-size: 13px; font-weight: 800; }\\n.sub { color: var(--muted); margin: 3px 0 0; font-size: 12px; }\\n.hint { color: var(--muted); margin: 6px 0 8px; font-size: 12px; min-height: 16px; }\\n\\n/* Chrome caps popups near 600px tall: once a result is showing, drop the explanatory lines. */\\nbody:has(#result:not(:empty)) .sub, body:has(#result:not(:empty)) .hint { display: none; }\\nbody:has(#result:not(:empty)) textarea { min-height: 44px; }\\n\\n/* segmented mode control */\\n.seg { display: flex; gap: 2px; padding: 3px; background: var(--card); border: 1px solid var(--line); border-radius: 10px; }\\n.seg button { flex: 1; padding: 5px 8px; border: 0; border-radius: 7px; background: transparent;\\n  color: var(--muted); font: inherit; font-weight: 600; cursor: pointer; }\\n.seg button[aria-checked=\\"true\\"] { background: var(--bg); color: var(--fg);\\n  box-shadow: 0 1px 2px rgba(0, 0, 0, .18), 0 0 0 1px var(--line); }\\n\\nlabel { display: block; font-weight: 600; margin: 8px 0 3px; font-size: 12px; }\\nselect, textarea, input:not([type=\\"hidden\\"]) { width: 100%; padding: 7px 9px; border: 1px solid var(--line);\\n  border-radius: 8px; background: var(--bg); color: var(--fg); font: inherit; }\\n#search { margin-bottom: 6px; }\\ntextarea { resize: vertical; min-height: 58px; }\\ntextarea:read-only { background: var(--card); }\\n\\n.tools { display: flex; align-items: center; gap: 6px; margin: 4px 0 10px; color: var(--muted); font-size: 12px; }\\nbutton.link { padding: 0; border: 0; background: none; color: var(--accent); font: inherit; font-size: 12px;\\n  cursor: pointer; }\\nbutton.link:hover { text-decoration: underline; }\\nbutton.primary { width: 100%; padding: 9px 12px; border-radius: 8px; border: 0; background: var(--accent);\\n  color: var(--accent-fg); font: inherit; font-weight: 700; cursor: pointer; }\\nbutton.primary:hover:not(:disabled) { filter: brightness(1.07); }\\nbutton:disabled { opacity: .55; cursor: default; }\\n:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }\\n\\n#result { margin-top: 12px; }\\n#result:empty { margin: 0; }\\n.card { background: var(--card); border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }\\n.badge { display: inline-block; font-size: 11px; padding: 1px 8px; border: 1px solid var(--line);\\n  border-radius: 999px; color: var(--muted); margin-bottom: 8px; background: var(--bg); }\\n.note { color: var(--muted); margin: 6px 0 0; font-size: 12px; }\\n\\n/* verdict banner */\\n.card.verdict-card { --vc: var(--nei); border-left: 4px solid var(--vc);\\n  background: color-mix(in srgb, var(--vc) 9%, var(--bg)); }\\n.verdict-card.Supported { --vc: var(--supported); }\\n.verdict-card.Refuted { --vc: var(--refuted); }\\n.verdict-card.NEI { --vc: var(--nei); }\\n.verdict-card.Conflicting { --vc: var(--conflict); }\\n.verdict { display: flex; align-items: center; gap: 8px; color: var(--vc); font-size: 17px; font-weight: 800; margin: 0 0 4px; }\\n.verdict .icon { display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 50%;\\n  background: var(--vc); color: var(--bg); font-size: 13px; line-height: 1; }\\n.claimline { color: var(--muted); font-size: 12px; margin: 0 0 6px; }\\n\\n.msg-warn { border-color: var(--warn); }\\n.msg-warn .title { color: var(--warn); font-weight: 700; }\\n.msg-error { border-color: var(--refuted); }\\n.msg-error .title { color: var(--refuted); font-weight: 700; }\\n\\nh2 { font-size: 11px; margin: 12px 0 6px; color: var(--muted); text-transform: uppercase; letter-spacing: .06em; }\\nol { list-style: none; margin: 0; padding: 0; }\\ndetails.src { border: 1px solid var(--line); border-radius: 10px; margin-bottom: 6px; background: var(--bg); }\\ndetails.src summary { list-style: none; cursor: pointer; padding: 8px 10px; }\\ndetails.src summary::-webkit-details-marker { display: none; }\\n.src-head { display: flex; align-items: center; gap: 6px; font-size: 11px; color: var(--muted); }\\n.src-rank { display: inline-grid; place-items: center; width: 18px; height: 18px; border-radius: 50%;\\n  background: var(--accent); color: var(--accent-fg); font-weight: 700; }\\n.src-domain { font-weight: 600; color: var(--fg); overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }\\n.src-chev { margin-left: auto; transition: transform .15s; }\\ndetails.src[open] .src-chev { transform: rotate(90deg); }\\n.preview { margin-top: 4px; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; }\\ndetails.src[open] .preview { display: none; }\\n.src-body { padding: 0 10px 10px; }\\n.src-body blockquote { margin: 0 0 6px; padding-left: 8px; border-left: 3px solid var(--line); max-height: 180px; overflow: auto; }\\n.src-body a { color: var(--accent); word-break: break-all; font-size: 12px; }\\n\\ndetails.settings { margin-top: 12px; }\\ndetails.settings summary { cursor: pointer; color: var(--muted); font-size: 12px; }\\n\\n.spinner { display: inline-block; width: 12px; height: 12px; margin-right: 6px; vertical-align: -2px;\\n  border: 2px solid var(--line); border-top-color: var(--accent); border-radius: 50%;\\n  animation: spin .8s linear infinite; }\\n@keyframes spin { to { transform: rotate(360deg); } }\\n@media (prefers-reduced-motion: reduce) { .spinner { animation: none; } .src-chev { transition: none; } }\\n", "apps/browser-extension/popup.html": "<!doctype html>\\n<html lang=\\"en\\">\\n<head>\\n  <meta charset=\\"utf-8\\">\\n  <meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1\\">\\n  <title>Claim Check</title>\\n  <link rel=\\"stylesheet\\" href=\\"popup.css\\">\\n</head>\\n<body>\\n  <header>\\n    <h1><span class=\\"logo\\" aria-hidden=\\"true\\">\\u2713</span>Claim Check</h1>\\n    <p class=\\"sub\\">A preliminary screen, not a final answer. You decide what to trust or share.</p>\\n  </header>\\n\\n  <section>\\n    <input type=\\"hidden\\" id=\\"mode\\" value=\\"mock\\">\\n    <div class=\\"seg\\" role=\\"radiogroup\\" aria-label=\\"Source of results\\">\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"mock\\" aria-checked=\\"true\\">Mock</button>\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"benchmark\\" aria-checked=\\"false\\">Benchmark</button>\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"live\\" aria-checked=\\"false\\">Live</button>\\n    </div>\\n    <p id=\\"modeHint\\" class=\\"hint\\"></p>\\n\\n    <div id=\\"modelRow\\" hidden>\\n      <label for=\\"model\\">Model / evidence pipeline</label>\\n      <select id=\\"model\\" aria-describedby=\\"modelHint\\" disabled></select>\\n      <p id=\\"modelHint\\" class=\\"hint\\">Connecting to the backend\\u2026</p>\\n    </div>\\n\\n    <div id=\\"scenarioRow\\" hidden>\\n      <label for=\\"scenario\\" id=\\"scenarioLabel\\">Sample outcome</label>\\n      <input id=\\"search\\" type=\\"search\\" placeholder=\\"Search demo claims\\" hidden>\\n      <select id=\\"scenario\\"></select>\\n    </div>\\n\\n    <label for=\\"claim\\">Claim</label>\\n    <textarea id=\\"claim\\" rows=\\"3\\" maxlength=\\"2000\\" placeholder=\\"Select text on a page, or type or paste a claim\\"></textarea>\\n    <div class=\\"tools\\">\\n      <button id=\\"useSelection\\" type=\\"button\\" class=\\"link\\">Use selected text</button>\\n      <span aria-hidden=\\"true\\">\\u00b7</span>\\n      <button id=\\"pasteClip\\" type=\\"button\\" class=\\"link\\">Paste from clipboard</button>\\n    </div>\\n    <button id=\\"check\\" type=\\"button\\" class=\\"primary\\">Check this claim</button>\\n  </section>\\n\\n  <section id=\\"result\\" aria-live=\\"polite\\"></section>\\n\\n  <details class=\\"settings\\">\\n    <summary>Backend settings</summary>\\n    <label for=\\"backend\\">Backend URL</label>\\n    <input id=\\"backend\\" value=\\"http://127.0.0.1:8000\\">\\n  </details>\\n\\n  <script src=\\"mock.js\\"></script>\\n  <script src=\\"popup.js\\"></script>\\n</body>\\n</html>\\n", "apps/browser-extension/popup.js": "\\"use strict\\";\\n\\nconst $ = id => document.getElementById(id);\\nconst els = { search: $(\\"search\\"), mode: $(\\"mode\\"), hint: $(\\"modeHint\\"), scenarioRow: $(\\"scenarioRow\\"), scenario: $(\\"scenario\\"),\\n  claim: $(\\"claim\\"), check: $(\\"check\\"), useSel: $(\\"useSelection\\"), result: $(\\"result\\"), backend: $(\\"backend\\"),\\n  model: $(\\"model\\"), modelRow: $(\\"modelRow\\"), modelHint: $(\\"modelHint\\") };\\n\\nconst HINTS = {\\n  mock: \\"Built-in samples for trying the interface. Not a real check.\\",\\n  benchmark: \\"Known benchmark claims only, served by the local backend. Not a check of arbitrary text.\\",\\n  live: \\"Checking new claims is not available yet; you will see an explanation instead of a result.\\",\\n};\\nconst MODE_BADGE = { mock: \\"Mock sample\\", benchmark: \\"Benchmark replay (saved)\\", live: \\"Live\\" };\\n// The backend says where a benchmark result came from; label it truthfully.\\nconst PROVENANCE_BADGE = {\\n  fixture_replay: \\"Benchmark replay (synthetic sample)\\",\\n  baseline_distilbert_bm25_top3: \\"Baseline model, benchmark claim (not a live check)\\",\\n};\\nconst VERDICT_CLASS = { \\"Supported\\": \\"Supported\\", \\"Refuted\\": \\"Refuted\\",\\n  \\"Not Enough Evidence\\": \\"NEI\\", \\"Conflicting Evidence\\": \\"Conflicting\\" };\\nconst VERDICT_ICON = { \\"Supported\\": \\"\\u2713\\", \\"Refuted\\": \\"\\u2715\\", \\"Not Enough Evidence\\": \\"?\\", \\"Conflicting Evidence\\": \\"\\u2194\\" };\\nconst MAX_SOURCES = 3;\\n\\nfunction domainOf(href) { try { return new URL(href).hostname.replace(/^www\\\\./, \\"\\"); } catch { return href; } }\\n\\nlet benchmarkClaims = []; // [{claim_id, claim}]\\nlet models = [], preferredModel = \\"distilbert_bm25\\", revision = 0, catalogRequest = 0, modelRequest = 0;\\n\\nasync function saveSettings() {\\n  const settings = { mode: els.mode.value, backend: els.backend.value, model: preferredModel };\\n  try {\\n    if (typeof chrome !== \\"undefined\\" && chrome.storage?.local) await chrome.storage.local.set({ claimCheckSettings: settings });\\n    else localStorage.setItem(\\"claimCheckSettings\\", JSON.stringify(settings));\\n  } catch { /* Preferences are optional; verification still works. */ }\\n}\\n\\nfunction clearResult() {\\n  revision++;\\n  els.result.replaceChildren();\\n  els.check.disabled = els.mode.value === \\"benchmark\\" && !models.some(m => m.id === els.model.value && m.available);\\n}\\n\\nfunction describeModel() {\\n  const picked = models.find(m => m.id === els.model.value);\\n  els.modelHint.textContent = picked ? picked.description : \\"No model available.\\";\\n}\\n\\nasync function loadModels() {\\n  const current = ++modelRequest, base = els.backend.value.replace(/\\\\/+$/, \\"\\");\\n  models = [];\\n  els.model.disabled = true;\\n  els.model.replaceChildren(new Option(\\"Loading models\\u2026\\", \\"\\"));\\n  els.check.disabled = true;\\n  try {\\n    const response = await fetch(base + \\"/models\\");\\n    if (!response.ok) throw new Error(\\"No model catalog\\");\\n    const data = await response.json();\\n    if (current !== modelRequest || els.mode.value !== \'benchmark\') return;\\n    if (!Array.isArray(data.models) || !data.models.length) throw new Error(\\"Empty model catalog\\");\\n    models = data.models;\\n    els.model.replaceChildren(...models.map(m => {\\n      const option = new Option(m.label + (m.available ? \\"\\" : \\" (unavailable)\\"), m.id);\\n      option.disabled = !m.available;\\n      option.title = m.unavailable_reason || m.description;\\n      return option;\\n    }));\\n    const chosen = models.find(m => m.id === preferredModel && m.available)\\n      || models.find(m => m.id === data.default_model_id && m.available);\\n    if (!chosen) throw new Error(\\"No model loaded\\");\\n    els.model.value = chosen.id;\\n    preferredModel = chosen.id;\\n    els.model.disabled = false;\\n    els.check.disabled = false;\\n    describeModel();\\n    saveSettings();\\n  } catch {\\n    if (current !== modelRequest || els.mode.value !== \'benchmark\') return;\\n    models = [];\\n    els.model.replaceChildren(new Option(\\"Backend unavailable\\", \\"\\"));\\n    els.modelHint.textContent = \\"Start or update the backend, then reopen Benchmark.\\";\\n  }\\n}\\n\\nfunction el(tag, cls, text) {\\n  const n = document.createElement(tag);\\n  if (cls) n.className = cls;\\n  if (text != null) n.textContent = text; // never innerHTML: claim/source text is untrusted\\n  return n;\\n}\\n\\nfunction safeHttpUrl(u) {\\n  try { const p = new URL(u); return (p.protocol === \\"https:\\" || p.protocol === \\"http:\\") ? p.href : null; }\\n  catch { return null; }\\n}\\n\\nfunction show(node) { els.result.replaceChildren(node); }\\n\\nfunction renderLoading() {\\n  const c = el(\\"div\\", \\"card\\");\\n  c.append(el(\\"span\\", \\"spinner\\"), document.createTextNode(\\"Checking\\u2026\\"));\\n  show(c);\\n}\\n\\nfunction renderMessage(kind, title, body, badge) {\\n  const c = el(\\"div\\", \\"card msg-\\" + kind);\\n  if (badge) c.append(el(\\"div\\", \\"badge\\", badge));\\n  c.append(el(\\"div\\", \\"title\\", title), el(\\"p\\", \\"note\\", body));\\n  show(c);\\n}\\n\\nfunction renderResponse(r) {\\n  const badge = r.mode === \\"benchmark\\" && r.model_label && r.provenance !== \\"fixture_replay\\"\\n    ? r.model_label + \\" \\u00b7 benchmark claim\\" : PROVENANCE_BADGE[r.provenance] || MODE_BADGE[r.mode] || \\"\\";\\n  if (r.status === \\"insufficient_sources\\") {\\n    return renderMessage(\\"warn\\", \\"Not enough sources to check this\\",\\n      (r.explanation || \\"\\") + (r.uncertainty_note ? \\" \\" + r.uncertainty_note : \\"\\"), badge);\\n  }\\n  if (r.status !== \\"completed\\" || !r.verdict) {\\n    return renderMessage(\\"error\\", \\"Something went wrong\\", r.explanation || \\"The check could not be completed.\\", badge);\\n  }\\n  const wrap = el(\\"div\\");\\n  const card = el(\\"div\\", \\"card verdict-card \\" + (VERDICT_CLASS[r.verdict] || \\"\\"));\\n  card.append(el(\\"div\\", \\"badge\\", badge));\\n  // The claim box already shows the claim in benchmark mode; repeat it only if it differs.\\n  if (r.claim && r.claim !== els.claim.value.trim()) card.append(el(\\"p\\", \\"claimline\\", \\"Claim checked: \\" + r.claim));\\n  const v = el(\\"div\\", \\"verdict\\");\\n  v.append(el(\\"span\\", \\"icon\\", VERDICT_ICON[r.verdict] || \\"\\u2022\\"), document.createTextNode(r.verdict));\\n  card.append(v, el(\\"div\\", null, r.explanation || \\"\\"));\\n  if (r.uncertainty_note) card.append(el(\\"p\\", \\"note\\", r.uncertainty_note));\\n  card.append(el(\\"p\\", \\"note\\", \\"Preliminary screening only. Read the sources and decide for yourself.\\"));\\n  wrap.append(card);\\n\\n  const sources = (r.evidence || []).slice().sort((a, b) => a.rank - b.rank).slice(0, MAX_SOURCES);\\n  if (sources.length) {\\n    wrap.append(el(\\"h2\\", null, \\"Sources, most relevant first\\"));\\n    const list = el(\\"ol\\");\\n    sources.forEach((s, i) => {\\n      const href = safeHttpUrl(s.url);\\n      const li = el(\\"li\\");\\n      const d = el(\\"details\\", \\"src\\");\\n      const sum = el(\\"summary\\");\\n      const head = el(\\"div\\", \\"src-head\\");\\n      head.append(el(\\"span\\", \\"src-rank\\", String(i + 1)), el(\\"span\\", \\"src-domain\\", href ? domainOf(href) : \\"Source link unavailable\\"),\\n        el(\\"span\\", \\"src-chev\\", \\"\\u203a\\"));\\n      sum.append(head, el(\\"div\\", \\"preview\\", s.text));\\n      const body = el(\\"div\\", \\"src-body\\");\\n      body.append(el(\\"blockquote\\", null, s.text));\\n      if (href) {\\n        const a = el(\\"a\\", null, href);\\n        a.href = href; a.target = \\"_blank\\"; a.rel = \\"noopener noreferrer\\";\\n        body.append(a);\\n      }\\n      d.append(sum, body);\\n      li.append(d);\\n      list.append(li);\\n    });\\n    wrap.append(list);\\n  }\\n  show(wrap);\\n}\\n\\nasync function postVerify(body) {\\n  const base = els.backend.value.replace(/\\\\/+$/, \\"\\");\\n  let resp;\\n  try {\\n    resp = await fetch(base + \\"/verify\\", { method: \\"POST\\", headers: { \\"Content-Type\\": \\"application/json\\" },\\n      body: JSON.stringify(body) });\\n  } catch {\\n    return { mode: body.mode, status: \\"error\\", verdict: null, evidence: [],\\n      explanation: \\"Could not reach the backend at \\" + base + \\". Is it running?\\" };\\n  }\\n  try { return await resp.json(); }\\n  catch { return { mode: body.mode, status: \\"error\\", verdict: null, evidence: [], explanation: \\"The backend sent an unreadable reply.\\" }; }\\n}\\n\\nasync function runCheck() {\\n  const current = ++revision;\\n  const mode = els.mode.value;\\n  const claim = els.claim.value.trim();\\n  if (!claim) return renderMessage(\\"warn\\", \\"Enter a claim first\\", \\"Select text on a page or type one above.\\");\\n  els.check.disabled = true;\\n  renderLoading();\\n  try {\\n    let r;\\n    if (mode === \\"mock\\") r = await mockVerify(claim, els.scenario.value);\\n    else if (mode === \\"live\\") r = await postVerify({ mode: \\"live\\", claim, page_url: null });\\n    else {\\n      const picked = benchmarkClaims.find(c => c.claim_id === els.scenario.value);\\n      r = picked ? await postVerify({ mode: \\"benchmark\\", claim_id: picked.claim_id, model_id: els.model.value })\\n        : { mode, status: \\"error\\", verdict: null, evidence: [], explanation: \\"Choose a demo claim.\\" };\\n    }\\n    if (current !== revision) return;\\n    renderResponse(r);\\n    els.result.scrollIntoView({ block: \\"start\\", behavior: \\"smooth\\" }); // popup is short: bring the verdict into view\\n  } finally { if (current === revision) els.check.disabled = false; }\\n}\\n\\nasync function loadBenchmarkClaims() {\\n  if (els.mode.value !== \\"benchmark\\") return;\\n  const current = ++catalogRequest;\\n  clearResult();\\n  els.claim.value = \\"\\";\\n  els.check.disabled = true;\\n  els.scenario.replaceChildren();\\n  let reachable = true;\\n  try {\\n    const base = els.backend.value.replace(/\\\\/+$/, \\"\\");\\n    const q = encodeURIComponent(els.search.value.trim());\\n    const data = await (await fetch(base + \\"/benchmark/claims?limit=50&q=\\" + q)).json();\\n    if (current !== catalogRequest || els.mode.value !== \'benchmark\') return;\\n    benchmarkClaims = data.claims || [];\\n  } catch {\\n    if (current !== catalogRequest || els.mode.value !== \'benchmark\') return;\\n    benchmarkClaims = []; reachable = false;\\n  }\\n  if (!reachable) {\\n    renderMessage(\\"error\\", \\"Backend not reachable\\", \\"Start it with: python3 apps/backend/server.py\\", MODE_BADGE.benchmark);\\n    return;\\n  }\\n  if (!benchmarkClaims.length) {\\n    els.claim.value = \\"\\";\\n    renderMessage(\\"warn\\", \\"No demo claims match\\", \\"Try a different search.\\");\\n    return;\\n  }\\n  benchmarkClaims.forEach(c => els.scenario.append(new Option(c.claim_id + \\" \\u00b7 \\" + c.claim.slice(0, 60), c.claim_id)));\\n  els.claim.value = benchmarkClaims[0].claim;\\n  clearResult();\\n}\\n\\nfunction applyMode() {\\n  modelRequest++;\\n  catalogRequest++;\\n  const mode = els.mode.value;\\n  els.hint.textContent = HINTS[mode];\\n  els.scenarioRow.hidden = mode === \\"live\\";\\n  els.search.hidden = mode !== \\"benchmark\\";\\n  els.claim.readOnly = mode === \\"benchmark\\"; // benchmark replays a fixed demo claim only\\n  els.claim.placeholder = mode === \\"benchmark\\" ? \\"Pick a demo claim above\\"\\n    : \\"Select text on a page, or type or paste a claim\\";\\n  $(\\"scenarioLabel\\").textContent = mode === \\"mock\\" ? \\"Sample outcome\\" : \\"Demo claim\\";\\n  clearResult();\\n  els.modelRow.hidden = mode !== \\"benchmark\\";\\n  document.querySelectorAll(\\".seg button\\").forEach(b => b.setAttribute(\\"aria-checked\\", String(b.dataset.mode === mode)));\\n  if (mode === \\"mock\\") {\\n    els.scenario.replaceChildren(...Object.entries(MOCK_SCENARIOS).map(([k, v]) => new Option(v.label, k)));\\n  } else if (mode === \\"benchmark\\") {\\n    loadModels();\\n    loadBenchmarkClaims();\\n  }\\n}\\n\\nconst NO_SELECTION_HELP = \\"Nothing could be read from the page. Some sites (for example Google Docs) \\" +\\n  \\"don\'t expose selected text to extensions. Copy the text, then use Paste from clipboard.\\";\\n\\nfunction benchmarkLocked() {\\n  if (els.mode.value !== \\"benchmark\\") return false;\\n  renderMessage(\\"warn\\", \\"Demo claims only\\", \\"Benchmark demo replays saved results for the listed demo claims. Switch to Mock or Live to use your own text.\\");\\n  return true;\\n}\\n\\nasync function useSelection(quiet) {\\n  if (els.mode.value === \\"benchmark\\") { if (!quiet) benchmarkLocked(); return; }\\n  if (typeof chrome === \\"undefined\\" || !chrome.tabs || !chrome.scripting) return;\\n  try {\\n    const [tab] = await chrome.tabs.query({ active: true, currentWindow: true });\\n    const results = await chrome.scripting.executeScript({ target: { tabId: tab.id, allFrames: true },\\n      func: () => String(window.getSelection()) });\\n    const text = results.map(r => (r.result || \\"\\").trim()).sort((a, b) => b.length - a.length)[0] || \\"\\";\\n    if (text) els.claim.value = text.slice(0, 2000);\\n    else if (!quiet) renderMessage(\\"warn\\", \\"No text selected\\", NO_SELECTION_HELP);\\n  } catch {\\n    if (!quiet) renderMessage(\\"warn\\", \\"Can\'t read this page\\", \\"Browser pages and the web store block this. Paste the claim instead.\\");\\n  }\\n}\\n\\nasync function pasteClipboard() {\\n  if (benchmarkLocked()) return;\\n  try {\\n    const text = (await navigator.clipboard.readText()).trim();\\n    if (text) { els.claim.value = text.slice(0, 2000); els.result.replaceChildren(); }\\n    else renderMessage(\\"warn\\", \\"Clipboard is empty\\", \\"Copy the claim text first.\\");\\n  } catch {\\n    renderMessage(\\"warn\\", \\"Clipboard not available\\", \\"Click in the Claim box and paste with Cmd/Ctrl+V.\\");\\n  }\\n}\\n\\ndocument.querySelectorAll(\\".seg button\\").forEach(b => b.addEventListener(\\"click\\", () => {\\n  if (els.mode.value === b.dataset.mode) return;\\n  els.mode.value = b.dataset.mode;\\n  document.querySelectorAll(\\".seg button\\").forEach(x => x.setAttribute(\\"aria-checked\\", String(x === b)));\\n  applyMode();\\n  saveSettings();\\n}));\\nlet searchTimer;\\nels.search.addEventListener(\\"input\\", () => { clearTimeout(searchTimer); searchTimer = setTimeout(loadBenchmarkClaims, 250); });\\nels.scenario.addEventListener(\\"change\\", () => {\\n  clearResult();\\n  if (els.mode.value !== \\"benchmark\\") return;\\n  const c = benchmarkClaims.find(x => x.claim_id === els.scenario.value);\\n  if (c) els.claim.value = c.claim;\\n});\\nels.model.addEventListener(\\"change\\", () => {\\n  preferredModel = els.model.value;\\n  clearResult();\\n  describeModel();\\n  saveSettings();\\n});\\nels.backend.addEventListener(\\"change\\", () => { applyMode(); saveSettings(); });\\nels.claim.addEventListener(\\"input\\", clearResult);\\nels.check.addEventListener(\\"click\\", runCheck);\\nels.useSel.addEventListener(\\"click\\", () => useSelection(false));\\n$(\\"pasteClip\\").addEventListener(\\"click\\", pasteClipboard);\\nasync function initialize() {\\n  try {\\n    const saved = typeof chrome !== \\"undefined\\" && chrome.storage?.local\\n      ? (await chrome.storage.local.get(\\"claimCheckSettings\\")).claimCheckSettings\\n      : JSON.parse(localStorage.getItem(\\"claimCheckSettings\\") || \\"null\\");\\n    if (saved) {\\n      if (HINTS[saved.mode]) els.mode.value = saved.mode;\\n      if (typeof saved.backend === \\"string\\") els.backend.value = saved.backend;\\n      if (typeof saved.model === \\"string\\") preferredModel = saved.model;\\n    }\\n  } catch { /* Start with defaults if preferences cannot be read. */ }\\n  applyMode();\\n  useSelection(true);\\n}\\ninitialize();\\n", "src/__init__.py": "\\"\\"\\"Misinformation screening assistant source package.\\"\\"\\"\\n", "src/data/__init__.py": "\\"\\"\\"Dataset loading and canonical record schemas.\\"\\"\\"\\n", "src/data/averitec.py": "\\"\\"\\"AVeriTeC claim loading, normalization, and audit helpers.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\n\\nfrom src.data.schema import ClaimRecord, EvidenceItem\\n\\nCANONICAL_LABELS = (\\n    \\"Supported\\",\\n    \\"Refuted\\",\\n    \\"Not Enough Evidence\\",\\n    \\"Conflicting Evidence\\",\\n)\\n\\n_LABEL_ALIASES = {\\n    \\"supported\\": \\"Supported\\",\\n    \\"refuted\\": \\"Refuted\\",\\n    \\"not enough evidence\\": \\"Not Enough Evidence\\",\\n    \\"conflicting evidence/cherrypicking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence/cherry-picking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence\\": \\"Conflicting Evidence\\",\\n}\\n\\n\\ndef normalize_label(label: str | None) -> str | None:\\n    \\"\\"\\"Return the project label corresponding to an official AVeriTeC label.\\"\\"\\"\\n    if label is None:\\n        return None\\n    cleaned = \\" \\".join(label.strip().split()).casefold()\\n    try:\\n        return _LABEL_ALIASES[cleaned]\\n    except KeyError as exc:\\n        raise ValueError(f\\"Unsupported AVeriTeC label: {label!r}\\") from exc\\n\\n\\ndef _read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:\\n    with path.open(\\"r\\", encoding=\\"utf-8\\") as handle:\\n        text = handle.read().strip()\\n    if not text:\\n        return []\\n    parsed = json.loads(text)\\n    if isinstance(parsed, list):\\n        return parsed\\n    if isinstance(parsed, dict):\\n        return [parsed]\\n    raise ValueError(f\\"Expected a JSON object or list in {path}\\")\\n\\n\\ndef _answers_to_evidence(questions: Iterable[dict[str, Any]], claim_id: str) -> tuple[EvidenceItem, ...]:\\n    \\"\\"\\"Expose annotated answers for gold-evidence verification only.\\n\\n    These records must never be used to construct the retrieval corpus.\\n    \\"\\"\\"\\n    evidence: list[EvidenceItem] = []\\n    for question_index, question in enumerate(questions):\\n        question_text = str(question.get(\\"question\\") or \\"\\").strip()\\n        for answer_index, answer in enumerate(question.get(\\"answers\\") or []):\\n            answer_text = str(answer.get(\\"answer\\") or \\"\\").strip()\\n            if not answer_text:\\n                continue\\n            source_url = str(answer.get(\\"source_url\\") or \\"\\").strip() or None\\n            evidence.append(\\n                EvidenceItem(\\n                    passage_id=f\\"{claim_id}:q{question_index}:a{answer_index}\\",\\n                    text=answer_text,\\n                    url=source_url,\\n                    metadata={\\n                        \\"question\\": question_text,\\n                        \\"answer_type\\": answer.get(\\"answer_type\\"),\\n                        \\"source_medium\\": answer.get(\\"source_medium\\"),\\n                        \\"is_gold_answer\\": True,\\n                    },\\n                )\\n            )\\n    return tuple(evidence)\\n\\n\\ndef raw_to_record(raw: dict[str, Any], split: str, index: int) -> ClaimRecord:\\n    \\"\\"\\"Convert one official AVeriTeC object into the project record shape.\\"\\"\\"\\n    claim = str(raw.get(\\"claim\\") or \\"\\").strip()\\n    if not claim:\\n        raise ValueError(f\\"{split}[{index}] has an empty claim\\")\\n    claim_id = str(raw.get(\\"claim_id\\") or f\\"averitec:{split}:{index:05d}\\")\\n    label = normalize_label(raw.get(\\"label\\"))\\n    questions = raw.get(\\"questions\\") or []\\n    if not isinstance(questions, list):\\n        raise ValueError(f\\"{claim_id} has a non-list questions field\\")\\n    source_urls = tuple(\\n        item.url for item in _answers_to_evidence(questions, claim_id) if item.url is not None\\n    )\\n    return ClaimRecord(\\n        claim_id=claim_id,\\n        claim=claim,\\n        label=label,\\n        source_urls=tuple(dict.fromkeys(source_urls)),\\n        evidence=_answers_to_evidence(questions, claim_id),\\n        metadata={\\n            \\"dataset\\": \\"AVeriTeC\\",\\n            \\"split\\": split,\\n            \\"source_index\\": index,\\n            \\"claim_date\\": raw.get(\\"claim_date\\"),\\n            \\"speaker\\": raw.get(\\"speaker\\"),\\n            \\"reporting_source\\": raw.get(\\"reporting_source\\"),\\n            \\"location_iso_code\\": raw.get(\\"location_ISO_code\\"),\\n            \\"required_reannotation\\": raw.get(\\"required_reannotation\\"),\\n            \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n        },\\n    )\\n\\n\\ndef load_split(path: str | Path, split: str) -> list[ClaimRecord]:\\n    \\"\\"\\"Load one official AVeriTeC split from a JSON file.\\"\\"\\"\\n    source_path = Path(path)\\n    records = [raw_to_record(raw, split, index) for index, raw in enumerate(_read_json_or_jsonl(source_path))]\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records\\n\\n\\ndef load_split_with_rejections(\\n    path: str | Path, split: str\\n) -> tuple[list[ClaimRecord], list[dict[str, Any]]]:\\n    \\"\\"\\"Load a split while preserving an audit trail for unusable source rows.\\n\\n    An empty claim cannot be evaluated by any project model, so it is excluded\\n    from prepared output. Any other schema or label error remains a hard failure\\n    until it is explicitly understood.\\n    \\"\\"\\"\\n    source_path = Path(path)\\n    records: list[ClaimRecord] = []\\n    rejections: list[dict[str, Any]] = []\\n    for index, raw in enumerate(_read_json_or_jsonl(source_path)):\\n        if not str(raw.get(\\"claim\\") or \\"\\").strip():\\n            rejections.append(\\n                {\\n                    \\"source_index\\": index,\\n                    \\"reason\\": \\"empty_claim\\",\\n                    \\"source_label\\": raw.get(\\"label\\"),\\n                    \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n                }\\n            )\\n            continue\\n        records.append(raw_to_record(raw, split, index))\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records, rejections\\n\\n\\ndef split_audit(records: Iterable[ClaimRecord]) -> dict[str, Any]:\\n    \\"\\"\\"Return a small, serializable audit report for a prepared split.\\"\\"\\"\\n    materialized = list(records)\\n    return {\\n        \\"record_count\\": len(materialized),\\n        \\"label_counts\\": dict(sorted(Counter(record.label for record in materialized).items())),\\n        \\"records_with_gold_answers\\": sum(bool(record.evidence) for record in materialized),\\n        \\"gold_answer_count\\": sum(len(record.evidence) for record in materialized),\\n    }\\n\\n\\ndef write_jsonl(records: Iterable[ClaimRecord], path: str | Path) -> None:\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    with destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for record in records:\\n            handle.write(json.dumps(record.to_dict(), ensure_ascii=False) + \\"\\\\n\\")\\n", "src/data/corpus.py": "\\"\\"\\"Candidate-document validation, passage chunking, and URL-based qrels.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom collections import Counter, defaultdict\\nfrom collections.abc import Iterable\\nfrom typing import Any\\nfrom urllib.parse import urlsplit, urlunsplit\\n\\n\\ndef canonical_url(value: str | None) -> str | None:\\n    \\"\\"\\"Normalize only URL details that should not change document identity.\\"\\"\\"\\n    if not value:\\n        return None\\n    raw = value.strip()\\n    if not raw:\\n        return None\\n    parts = urlsplit(raw)\\n    if not parts.scheme or not parts.netloc:\\n        return raw.rstrip(\\"/\\")\\n    return urlunsplit((parts.scheme.casefold(), parts.netloc.casefold(), parts.path.rstrip(\\"/\\"), parts.query, \\"\\"))\\n\\n\\ndef chunk_text(text: str, max_words: int = 160, overlap_words: int = 40) -> list[str]:\\n    \\"\\"\\"Split a document into deterministic overlapping word passages.\\"\\"\\"\\n    if max_words < 1 or overlap_words < 0 or overlap_words >= max_words:\\n        raise ValueError(\\"Require max_words > 0 and 0 <= overlap_words < max_words\\")\\n    words = re.findall(r\\"\\\\S+\\", text)\\n    if not words:\\n        return []\\n    passages = []\\n    step = max_words - overlap_words\\n    for start in range(0, len(words), step):\\n        passage = \\" \\".join(words[start : start + max_words]).strip()\\n        if passage:\\n            passages.append(passage)\\n        if start + max_words >= len(words):\\n            break\\n    return passages\\n\\n\\ndef documents_to_passages(\\n    documents: Iterable[dict[str, Any]], max_words: int = 160, overlap_words: int = 40\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Convert normalized candidate documents into provenance-preserving passages.\\n\\n    Input documents must have a non-empty `text` and should have a stable `url`\\n    or `document_id`. This function is intentionally format-agnostic; the\\n    AVeriTeC archive inspector identifies how to create these documents.\\n    \\"\\"\\"\\n    passages: list[dict[str, Any]] = []\\n    rejected = 0\\n    for document_index, document in enumerate(documents):\\n        text = str(document.get(\\"text\\") or \\"\\").strip()\\n        if not text:\\n            rejected += 1\\n            continue\\n        document_id = str(document.get(\\"document_id\\") or document.get(\\"url\\") or f\\"document:{document_index}\\")\\n        url = document.get(\\"url\\")\\n        for passage_index, passage_text in enumerate(chunk_text(text, max_words, overlap_words)):\\n            passages.append(\\n                {\\n                    \\"passage_id\\": f\\"{document_id}:p{passage_index:04d}\\",\\n                    \\"text\\": passage_text,\\n                    \\"url\\": url,\\n                    \\"metadata\\": {\\n                        \\"document_id\\": document_id,\\n                        \\"chunk_index\\": passage_index,\\n                        \\"chunk_max_words\\": max_words,\\n                        \\"chunk_overlap_words\\": overlap_words,\\n                    },\\n                }\\n            )\\n    return passages, {\\"passage_count\\": len(passages), \\"rejected_empty_document_count\\": rejected}\\n\\n\\ndef build_url_qrels(\\n    claims: Iterable[dict[str, Any]], passages: Iterable[dict[str, Any]]\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Build *source-URL* relevance judgments from AVeriTeC annotations.\\n\\n    This is an auditable weak passage-level proxy: every passage from a human\\n    annotated evidence URL is marked relevant. Reports must call it URL-based\\n    evidence coverage, not exact human passage relevance.\\n    \\"\\"\\"\\n    # Claims are compact; materialize them once.  The large corpus itself is\\n    # streamed and only passage IDs for annotated URLs are retained.\\n    claim_rows = list(claims)\\n    annotated_urls = {\\n        canonical_url(url)\\n        for claim in claim_rows\\n        for url in (claim.get(\\"source_urls\\") or [])\\n    }\\n    annotated_urls.discard(None)\\n    passage_ids_by_url: dict[str, list[tuple[str, str | None]]] = defaultdict(list)\\n    for passage in passages:\\n        url = canonical_url(passage.get(\\"url\\"))\\n        if url in annotated_urls:\\n            candidate_claim_id = passage.get(\\"metadata\\", {}).get(\\"candidate_claim_id\\")\\n            passage_ids_by_url[url].append((str(passage[\\"passage_id\\"]), str(candidate_claim_id) if candidate_claim_id else None))\\n\\n    qrels: list[dict[str, Any]] = []\\n    unresolved_url_count = 0\\n    for claim in claim_rows:\\n        gold_urls = {canonical_url(url) for url in claim.get(\\"source_urls\\") or []}\\n        gold_urls.discard(None)\\n        relevant_ids = sorted(\\n            {\\n                passage_id\\n                for url in gold_urls\\n                for passage_id, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n                if candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n            }\\n        )\\n        unresolved = sorted(\\n            url\\n            for url in gold_urls\\n            if not any(\\n                candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n                for _, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n            )\\n        )\\n        unresolved_url_count += len(unresolved)\\n        qrels.append(\\n            {\\n                \\"claim_id\\": claim[\\"claim_id\\"],\\n                \\"relevant_passage_ids\\": relevant_ids,\\n                \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n                \\"annotated_source_url_count\\": len(gold_urls),\\n                \\"unresolved_source_urls\\": unresolved,\\n            }\\n        )\\n    audit = {\\n        \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n        \\"claim_count\\": len(qrels),\\n        \\"judged_claim_count\\": sum(bool(row[\\"relevant_passage_ids\\"]) for row in qrels),\\n        \\"unresolved_annotated_source_url_count\\": unresolved_url_count,\\n        \\"relevant_passage_count_distribution\\": dict(\\n            sorted(Counter(len(row[\\"relevant_passage_ids\\"]) for row in qrels).items())\\n        ),\\n    }\\n    return qrels, audit\\n", "src/data/schema.py": "\\"\\"\\"Shared, serializable records used throughout the project.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\n@dataclass(frozen=True)\\nclass EvidenceItem:\\n    \\"\\"\\"A retrievable evidence passage with its source provenance.\\"\\"\\"\\n\\n    passage_id: str\\n    text: str\\n    url: str | None = None\\n    domain: str | None = None\\n    score: float | None = None\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        return asdict(self)\\n\\n\\n@dataclass(frozen=True)\\nclass ClaimRecord:\\n    \\"\\"\\"Canonical claim record emitted by every dataset loader.\\"\\"\\"\\n\\n    claim_id: str\\n    claim: str\\n    label: str | None\\n    source_urls: tuple[str, ...] = ()\\n    evidence: tuple[EvidenceItem, ...] = ()\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        data = asdict(self)\\n        data[\\"source_urls\\"] = list(self.source_urls)\\n        return data\\n", "src/evaluation/__init__.py": "\\"\\"\\"Shared evaluation functions for project baselines and models.\\"\\"\\"\\n", "src/evaluation/splits.py": "\\"\\"\\"Train-only holdouts for diagnostics; never uses official dev claims.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom datetime import datetime\\nfrom math import ceil\\nimport re\\nimport unicodedata\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\\n\\n\\ndef parse_claim_date(value: Any) -> str | None:\\n    if not value:\\n        return None\\n    text = str(value).strip()\\n    for fmt in (\\"%Y-%m-%d\\", \\"%d-%m-%Y\\", \\"%d/%m/%Y\\", \\"%Y/%m/%d\\", \\"%m/%d/%Y\\", \\"%B %d, %Y\\"):\\n        try:\\n            return datetime.strptime(text[:10] if fmt == \\"%Y-%m-%d\\" else text, fmt).date().isoformat()\\n        except ValueError:\\n            pass\\n    return None\\n\\n\\ndef selection_split(records: list[dict], fraction: float, seed: int, policy: str = \\"stratified\\") -> tuple[np.ndarray, np.ndarray]:\\n    indices = np.arange(len(records))\\n    if policy == \\"stratified\\":\\n        return train_test_split(indices, test_size=fraction, random_state=seed, stratify=[r[\\"label\\"] for r in records])\\n    if policy == \\"grouped\\":\\n        groups = [\\" \\".join(re.findall(r\\"\\\\w+\\", unicodedata.normalize(\\"NFKC\\", r[\\"claim\\"]).casefold())) for r in records]\\n        labels = [r[\\"label\\"] for r in records]\\n        folds = min(max(2, round(1 / fraction)), min(labels.count(label) for label in set(labels)))\\n        if folds < 2:\\n            raise ValueError(\\"Too few examples for a grouped train holdout\\")\\n        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)\\n        return next(splitter.split(indices, labels, groups))\\n    if policy != \\"temporal\\":\\n        raise ValueError(f\\"Unknown validation policy: {policy}\\")\\n    dates = [parse_claim_date(r.get(\\"metadata\\", {}).get(\\"claim_date\\")) for r in records]\\n    if any(date is None for date in dates):\\n        raise ValueError(\\"Temporal holdout requires a parseable claim_date for every train claim; audit missing dates first\\")\\n    ordered = sorted(indices, key=lambda i: (dates[i], str(records[i][\\"claim_id\\"])))\\n    boundary = dates[ordered[max(1, len(records) - ceil(fraction * len(records)))]]\\n    train = np.asarray([i for i in ordered if dates[i] < boundary], dtype=int)\\n    holdout = np.asarray([i for i in ordered if dates[i] >= boundary], dtype=int)\\n    if not len(train) or not len(holdout):\\n        raise ValueError(\\"Temporal dates do not admit a nonempty earlier/later split\\")\\n    return train, holdout\\n", "src/retrieval/__init__.py": "\\"\\"\\"Evidence retrieval implementations and metrics.\\"\\"\\"\\n", "src/retrieval/evidence_quality.py": "\\"\\"\\"Prediction-time evidence hygiene and stance-aware selection.\\n\\nNever consumes verdicts, annotated answers, source_urls, qrels, or the original\\nfact-check URL. Unknown publication dates stay unknown. Future years in prose\\nare flagged, not treated as proof of publication after the claim.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nimport unicodedata\\nfrom dataclasses import asdict, dataclass\\nfrom urllib.parse import urlparse\\n\\nimport numpy as np\\n\\nfrom src.data.corpus import canonical_url\\nfrom src.evaluation.splits import parse_claim_date\\nfrom src.retrieval.question_guided import words\\n\\nQUALITY_NAMES = (\\"query_coverage\\", \\"number_overlap\\", \\"rank_prior\\", \\"satire_source\\", \\"factcheck_source\\", \\"publication_known\\", \\"future_year_mentioned\\", \\"search_results_source\\")\\nSATIRE_HOSTS = {\\"babylonbee.com\\", \\"theonion.com\\"}\\n\\n\\n@dataclass(frozen=True)\\nclass EvidencePolicy:\\n    max_candidates: int = 24\\n    candidate_depth: int = 100\\n    near_duplicate_threshold: float = 0.90\\n    remove_search_pages: bool = True\\n    remove_known_future_publications: bool = True\\n\\n    def to_dict(self):\\n        return asdict(self)\\n\\n\\ndef original_url(url: str | None) -> str:\\n    value = str(url or \\"\\")\\n    # Capture time is deliberately not used as publication time.\\n    match = re.match(r\\"https?://web\\\\.archive\\\\.org/web/\\\\d+(?:[a-z_]+)?/(https?://.*)\\", value, re.I)\\n    return match.group(1) if match else value\\n\\n\\ndef source_host(url: str | None) -> str:\\n    return (urlparse(original_url(url)).hostname or \\"\\").lower().removeprefix(\\"www.\\")\\n\\n\\ndef is_search_page(url: str | None) -> bool:\\n    parsed = urlparse(original_url(url))\\n    host = (parsed.hostname or \\"\\").lower().removeprefix(\\"www.\\")\\n    return bool((re.fullmatch(r\\"google\\\\.[a-z.]+\\", host) and parsed.path.startswith((\\"/search\\", \\"/url\\")))\\n                or (host in {\\"bing.com\\", \\"search.yahoo.com\\", \\"duckduckgo.com\\"} and (parsed.query or \\"/search\\" in parsed.path)))\\n\\n\\ndef publication_date(item: dict) -> str | None:\\n    metadata = item.get(\\"metadata\\") or {}\\n    for key in (\\"publication_date\\", \\"published_date\\", \\"published_at\\", \\"datePublished\\"):\\n        date = parse_claim_date(metadata.get(key) or item.get(key))\\n        if date:\\n            return date\\n    # URL paths and Wayback captures do not establish publication dates.\\n    return None\\n\\n\\ndef normalized_text(text: str) -> str:\\n    return \\" \\".join(re.findall(r\\"\\\\w+\\", unicodedata.normalize(\\"NFKC\\", text).casefold()))\\n\\n\\ndef shingles(text: str) -> set[tuple[str, ...]]:\\n    tokens = normalized_text(text).split()\\n    return {tuple(tokens[i:i + 5]) for i in range(max(1, len(tokens) - 4))}\\n\\n\\ndef quality_features(claim: str, item: dict, rank: int, claim_date: str | None) -> list[float]:\\n    text = str(item.get(\\"text\\") or \\"\\")\\n    query, terms = words(claim), words(text)\\n    numbers = set(re.findall(r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*\\\\b\\", claim))\\n    passage_numbers = set(re.findall(r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*\\\\b\\", text))\\n    host, url = source_host(item.get(\\"url\\")), original_url(item.get(\\"url\\")).lower()\\n    year = int(claim_date[:4]) if claim_date else None\\n    later = bool(year and any(int(y) > year for y in re.findall(r\\"\\\\b(20\\\\d{2})\\\\b\\", text)))\\n    return [len(query & terms) / max(1, len(query)), len(numbers & passage_numbers) / max(1, len(numbers)),\\n            1.0 / math.log2(rank + 2), float(host in SATIRE_HOSTS),\\n            float(bool(re.search(r\\"fact[-_]?check\\", url))), float(publication_date(item) is not None), float(later), float(is_search_page(item.get(\\"url\\")))]\\n\\n\\ndef pair_premise(item: dict) -> str:\\n    host = source_host(item.get(\\"url\\")) or \\"unknown\\"\\n    kind = \\"satire source\\" if host in SATIRE_HOSTS else \\"search results\\" if is_search_page(item.get(\\"url\\")) else \\"unspecified\\"\\n    return f\\"Source domain: {host}. Source type: {kind}.\\\\nPassage: {item.get(\'text\') or \'\'}\\"\\n\\n\\ndef clean_candidates(claim: str, claim_date: str | None, candidates: list[dict], policy: EvidencePolicy) -> tuple[list[dict], list[dict]]:\\n    date = parse_claim_date(claim_date)\\n    selected, rejected, signatures, seen_ids = [], [], [], set()\\n    if policy.max_candidates < 1 or policy.candidate_depth < 1 or not 0 < policy.near_duplicate_threshold <= 1:\\n        raise ValueError(\\"Invalid evidence policy\\")\\n    for rank, source in enumerate(candidates[:policy.candidate_depth]):\\n        pid = str(source.get(\\"passage_id\\") or \\"\\")\\n        if not pid or pid in seen_ids:\\n            raise ValueError(f\\"Missing/duplicate candidate passage ID: {pid!r}\\")\\n        seen_ids.add(pid)\\n        text = str(source.get(\\"text\\") or \\"\\").strip()\\n        reason = \\"blank\\" if not text else None\\n        if policy.remove_search_pages and is_search_page(source.get(\\"url\\")):\\n            reason = \\"search_results_page\\"\\n        published = publication_date(source)\\n        if policy.remove_known_future_publications and date and published and published > date:\\n            reason = \\"published_after_claim\\"\\n        signature = shingles(text)\\n        if reason is None:\\n            for old_pid, old_signature in signatures:\\n                ratio = len(signature & old_signature) / max(1, len(signature | old_signature))\\n                if ratio >= policy.near_duplicate_threshold:\\n                    reason = f\\"duplicate_of:{old_pid}\\"\\n                    break\\n        if reason:\\n            rejected.append({\\"passage_id\\": pid, \\"reason\\": reason, \\"original_rank\\": rank + 1})\\n            continue\\n        signatures.append((pid, signature))\\n        # Whitelist only fields that exist before a verdict is annotated.\\n        item = {\\"passage_id\\": pid, \\"text\\": text, \\"url\\": source.get(\\"url\\"), \\"original_rank\\": rank + 1,\\n                \\"publication_date\\": published, \\"quality\\": quality_features(claim, source, rank + 1, date)}\\n        selected.append(item)\\n        if len(selected) == policy.max_candidates:\\n            break\\n    return selected, rejected\\n\\n\\ndef nli_label_order(id2label: dict, explicit: str | None = None) -> list[str]:\\n    names = explicit.split(\\",\\") if explicit else [str(id2label.get(i, id2label.get(str(i), \\"\\"))).lower() for i in range(3)]\\n    names = [n.strip().lower() for n in names]\\n    if len(names) != 3 or set(names) != {\\"entailment\\", \\"neutral\\", \\"contradiction\\"}:\\n        raise ValueError(\\"Cannot identify NLI labels. Supply --nli-label-order in the model\'s documented logit order.\\")\\n    return names\\n\\n\\ndef stance_rank(item: dict) -> float:\\n    p = item[\\"nli_probabilities\\"]\\n    q = item[\\"quality\\"]\\n    # Both support and refutation are valuable; neither is a claim verdict.\\n    return 0.60 * (1 - p[\\"neutral\\"]) + 0.25 * q[0] + 0.15 * q[2] - 0.20 * q[3]\\n\\n\\ndef select_evidence(candidates: list[dict], *, rerank: bool) -> list[dict]:\\n    order = sorted(candidates, key=lambda p: (-stance_rank(p), p[\\"original_rank\\"], p[\\"passage_id\\"])) if rerank else list(candidates)\\n    chosen, urls = [], set()\\n    for p in order:\\n        source = canonical_url(original_url(p.get(\\"url\\"))) or p[\\"passage_id\\"]\\n        if rerank and source in urls:\\n            continue\\n        chosen.append(p)\\n        urls.add(source)\\n        if len(chosen) == 3:\\n            return chosen\\n    # Low source count: use additional distinct passages, never deleted garbage.\\n    used = {p[\\"passage_id\\"] for p in chosen}\\n    return (chosen + [p for p in order if p[\\"passage_id\\"] not in used])[:3]\\n\\n\\ndef aggregate_features(row: dict) -> np.ndarray:\\n    \\"\\"\\"Cheap learned-aggregation baseline using the same selected three passages.\\"\\"\\"\\n    evidence = row.get(\\"retrieved\\") or []\\n    p = np.asarray([[item[\\"nli_probabilities\\"][k] for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")] for item in evidence[:3]], dtype=float)\\n    q = np.asarray([item[\\"quality\\"] for item in evidence[:3]], dtype=float)\\n    if not len(p):\\n        return np.zeros(3 * 2 + len(QUALITY_NAMES) * 2 + 2, dtype=float)\\n    return np.concatenate([p.mean(0), p.max(0), q.mean(0), q.max(0), [p[:, 0].max() * p[:, 1].max(), len(p) / 3]])\\n", "src/retrieval/question_guided.py": "\\"\\"\\"Pure, testable parts of question-guided evidence selection.\\n\\nNo labels, gold answers, or qrels are inputs to these functions.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport re\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\n\\n\\nTOKEN = re.compile(r\\"[a-z0-9]+\\", re.IGNORECASE)\\nSENTENCE_BOUNDARY = re.compile(r\\"(?<=[.!?])\\\\s+(?=[A-Z0-9\\\\\\"\'([])\\")\\nSTOPWORDS = {\\n    \\"a\\", \\"an\\", \\"and\\", \\"are\\", \\"as\\", \\"at\\", \\"be\\", \\"by\\", \\"did\\", \\"do\\", \\"does\\", \\"for\\", \\"from\\",\\n    \\"has\\", \\"have\\", \\"how\\", \\"in\\", \\"is\\", \\"it\\", \\"of\\", \\"on\\", \\"or\\", \\"that\\", \\"the\\", \\"their\\",\\n    \\"there\\", \\"this\\", \\"to\\", \\"was\\", \\"were\\", \\"what\\", \\"when\\", \\"where\\", \\"which\\", \\"who\\", \\"why\\",\\n    \\"with\\", \\"would\\", \\"could\\", \\"should\\", \\"claim\\", \\"evidence\\", \\"supports\\", \\"contradicts\\",\\n}\\n\\n\\ndef words(text: str) -> set[str]:\\n    return set(TOKEN.findall(text.lower())) - STOPWORDS\\n\\n\\ndef parse_questions(response: str, claim: str, *, limit: int = 2) -> list[str]:\\n    \\"\\"\\"Accept JSON questions only; fall back to a claim-derived question.\\"\\"\\"\\n    found: list[str] = []\\n    decoder = json.JSONDecoder()\\n    for index, char in enumerate(response):\\n        if char != \\"{\\":\\n            continue\\n        try:\\n            value, _ = decoder.raw_decode(response[index:])\\n        except json.JSONDecodeError:\\n            continue\\n        if isinstance(value, dict) and isinstance(value.get(\\"questions\\"), list):\\n            for item in value[\\"questions\\"]:\\n                if isinstance(item, str):\\n                    question = \\" \\".join(item.split())[:240]\\n                    if 8 <= len(question) and question.lower() not in {q.lower() for q in found}:\\n                        found.append(question)\\n                if len(found) == limit:\\n                    break\\n            break\\n    if len(found) < limit:\\n        return [\\n            f\\"What evidence supports or contradicts this claim: {claim[:180]}?\\",\\n            f\\"Which source verifies the date, number, or entity in this claim: {claim[:180]}?\\",\\n        ][:limit]\\n    return found\\n\\n\\ndef shortlist_candidates(row: dict[str, Any], questions: list[str], *, limit: int, bm25_keep: int) -> list[dict[str, Any]]:\\n    \\"\\"\\"Retain BM25 leaders and candidates matching the question terms.\\"\\"\\"\\n    candidates = list(row.get(\\"retrieved\\") or [])\\n    ids = [str(item[\\"passage_id\\"]) for item in candidates]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate candidate passage IDs for {row.get(\'claim_id\')}\\")\\n    if limit < 1 or bm25_keep < 0:\\n        raise ValueError(\\"limit must be positive and bm25_keep nonnegative\\")\\n    query_terms = words(\\" \\".join([str(row.get(\\"claim\\") or \\"\\"), *questions]))\\n    leader_indices = list(range(min(bm25_keep, limit, len(candidates))))\\n    scored = []\\n    for index, item in enumerate(candidates):\\n        overlap = len(query_terms & words(str(item.get(\\"text\\") or \\"\\")[:1600]))\\n        scored.append((-overlap, index))\\n    chosen = set(leader_indices)\\n    for _, index in sorted(scored):\\n        if len(chosen) >= min(limit, len(candidates)):\\n            break\\n        chosen.add(index)\\n    return [candidates[index] for index in sorted(chosen)]\\n\\n\\ndef reciprocal_rank_scores(score_matrix: list[list[float]], *, claim_weight: float = 1.0) -> list[float]:\\n    \\"\\"\\"Fuse per-query cross-encoder ranks without comparing raw logits across queries.\\"\\"\\"\\n    if not score_matrix:\\n        return []\\n    size = len(score_matrix[0])\\n    if any(len(row) != size or any(not math.isfinite(float(value)) for value in row) for row in score_matrix):\\n        raise ValueError(\\"Cross-encoder score matrix is ragged or non-finite\\")\\n    result = [0.0] * size\\n    for query_index, scores in enumerate(score_matrix):\\n        weight = claim_weight if query_index == 0 else 1.0\\n        for rank, index in enumerate(sorted(range(size), key=lambda i: (-scores[i], i)), start=1):\\n            result[index] += weight / (20 + rank)\\n    return result\\n\\n\\ndef select_diverse(candidates: list[dict[str, Any]], scores: list[float], *, k: int = 3) -> list[dict[str, Any]]:\\n    \\"\\"\\"Select at most k unique-source passages, then fill if source count is low.\\"\\"\\"\\n    if len(candidates) != len(scores):\\n        raise ValueError(\\"Candidate and score counts differ\\")\\n    order = sorted(range(len(candidates)), key=lambda i: (-scores[i], str(candidates[i][\\"passage_id\\"])))\\n    selected: list[int] = []\\n    seen_sources: set[str] = set()\\n    for index in order:\\n        item = candidates[index]\\n        url = canonical_url(item.get(\\"url\\"))\\n        source = f\\"url:{url}\\" if url else f\\"passage:{item[\'passage_id\']}\\"\\n        if source not in seen_sources:\\n            selected.append(index)\\n            seen_sources.add(source)\\n        if len(selected) == k:\\n            break\\n    if len(selected) < k:\\n        selected.extend(index for index in order if index not in selected)\\n    return [{**candidates[index], \\"score\\": float(scores[index])} for index in selected[:k]]\\n\\n\\ndef extract_sentences(text: str, claim: str, questions: list[str], *, max_chars: int = 900, max_sentences: int = 3) -> str:\\n    \\"\\"\\"Select exact source sentences by term coverage; never generate evidence text.\\"\\"\\"\\n    original = str(text).strip()\\n    if len(original) <= max_chars:\\n        return original\\n    sentences = [item.strip() for item in SENTENCE_BOUNDARY.split(original) if item.strip()]\\n    if len(sentences) < 2:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    query_sets = [words(claim), *(words(q) for q in questions)]\\n    scored = []\\n    for index, sentence in enumerate(sentences):\\n        tokens = words(sentence)\\n        coverage = sum(len(tokens & query) / max(1, len(query)) for query in query_sets)\\n        scored.append((-coverage, index))\\n    chosen: list[int] = []\\n    used = 0\\n    for _, index in sorted(scored):\\n        sentence = sentences[index]\\n        if used + len(sentence) + (1 if chosen else 0) > max_chars:\\n            continue\\n        chosen.append(index)\\n        used += len(sentence) + (1 if len(chosen) > 1 else 0)\\n        if len(chosen) == max_sentences:\\n            break\\n    if not chosen:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    return \\" \\".join(sentences[index] for index in sorted(chosen))\\n", "src/utils/__init__.py": "\\"\\"\\"Small shared utilities for reproducible experiment scripts.\\"\\"\\"\\n", "src/utils/progress.py": "\\"\\"\\"Optional notebook-friendly progress bars with a dependency-free fallback.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\nclass _NoOpProgress:\\n    def update(self, _: int | float = 1) -> None:\\n        return None\\n\\n    def set_postfix(self, **_: Any) -> None:\\n        return None\\n\\n    def __enter__(self) -> \\"_NoOpProgress\\":\\n        return self\\n\\n    def __exit__(self, *_: Any) -> None:\\n        return None\\n\\n\\ndef progress(*, total: int | None, description: str, unit: str = \\"item\\", **kwargs: Any) -> Any:\\n    \\"\\"\\"Return `tqdm.auto` when installed; scripts remain usable without it.\\"\\"\\"\\n    try:\\n        from tqdm.auto import tqdm\\n    except ImportError:\\n        return _NoOpProgress()\\n    return tqdm(total=total, desc=description, unit=unit, **kwargs)\\n", "src/verification/__init__.py": "\\"\\"\\"Claim-verification baselines and later learned verifiers.\\"\\"\\"\\n", "src/verification/pair_reader.py": "\\"\\"\\"NLI-initialized, permutation-invariant three-passage verdict reader.\\n\\nFour-way verdict supervision applies to the aggregate, never to every passage.\\nA small distillation term retains the pretrained three-way NLI behavior using\\ncached teacher probabilities computed from prediction-time text alone.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nfrom torch import nn\\nfrom transformers import AutoConfig, AutoModelForSequenceClassification, PretrainedConfig, PreTrainedModel\\nfrom transformers.modeling_outputs import SequenceClassifierOutput\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.retrieval.evidence_quality import QUALITY_NAMES, nli_label_order, pair_premise\\n\\n\\ndef weighted_verdict_loss(logits, labels, weights, normalization=\\"sample_mean\\"):\\n    \\"\\"\\"Keep class weights effective even for singleton/single-class microbatches.\\n\\n    Weights have dataset mean one. Averaging weighted per-example losses gives\\n    the same gradient under equal-size microbatch accumulation as a full batch.\\n    The legacy option is retained for old checkpoints and controlled ablations.\\n    \\"\\"\\"\\n    if normalization not in (\\"sample_mean\\", \\"batch_weight_mean\\"):\\n        raise ValueError(f\\"Unknown loss normalization: {normalization}\\")\\n    reduction = \\"none\\" if normalization == \\"sample_mean\\" else \\"mean\\"\\n    loss = nn.functional.cross_entropy(logits.float(), labels, weight=weights.float(), reduction=reduction)\\n    return loss.mean()\\n\\n\\nclass PairReaderConfig(PretrainedConfig):\\n    model_type = \\"evidence_pair_reader\\"\\n\\n    def __init__(self, encoder_config=None, projection_size=192, dropout=0.15,\\n                 nli_labels=None, class_weights=None, distillation_weight=0.1, freeze_word_embeddings=True,\\n                 loss_normalization=\\"batch_weight_mean\\", **kwargs):\\n        kwargs.pop(\\"num_labels\\", None)\\n        super().__init__(num_labels=4, **kwargs)\\n        self.encoder_config = encoder_config or {}\\n        self.projection_size = projection_size\\n        self.dropout = dropout\\n        self.nli_labels = nli_labels or [\\"contradiction\\", \\"entailment\\", \\"neutral\\"]\\n        self.class_weights = class_weights or [1.0] * 4\\n        self.distillation_weight = distillation_weight\\n        self.freeze_word_embeddings = freeze_word_embeddings\\n        if loss_normalization not in (\\"sample_mean\\", \\"batch_weight_mean\\"):\\n            raise ValueError(\\"Unknown verdict-loss normalization\\")\\n        self.loss_normalization = loss_normalization\\n        self.id2label = dict(enumerate(CANONICAL_LABELS))\\n        self.label2id = {v: k for k, v in self.id2label.items()}\\n\\n\\nclass EvidencePairReader(PreTrainedModel):\\n    config_class = PairReaderConfig\\n    base_model_prefix = \\"reader\\"\\n    accepts_loss_kwargs = False\\n\\n    def __init__(self, config):\\n        super().__init__(config)\\n        encoder = dict(config.encoder_config)\\n        kind = encoder.pop(\\"model_type\\")\\n        self.reader = AutoModelForSequenceClassification.from_config(AutoConfig.for_model(kind, **encoder))\\n        hidden = int(self.reader.config.hidden_size)\\n        size, q = config.projection_size, len(QUALITY_NAMES)\\n        self.project = nn.Sequential(nn.Linear(hidden + 3 + q, size), nn.GELU(), nn.LayerNorm(size))\\n        self.attention = nn.Linear(size, 1)\\n        self.verdict = nn.Sequential(nn.Dropout(config.dropout), nn.Linear(2 * size + q + 8, size),\\n                                     nn.GELU(), nn.Dropout(config.dropout), nn.Linear(size, 4))\\n        self.register_buffer(\\"verdict_weights\\", torch.tensor(config.class_weights, dtype=torch.float32))\\n        self.post_init()\\n        if config.freeze_word_embeddings:\\n            self.reader.get_input_embeddings().requires_grad_(False)\\n\\n    @classmethod\\n    def from_nli(cls, name, revision=None, *, class_weights=None, label_order=None, distillation_weight=0.1,\\n                 freeze_word_embeddings=True, loss_normalization=\\"sample_mean\\"):\\n        reader = AutoModelForSequenceClassification.from_pretrained(name, revision=revision)\\n        if int(reader.config.num_labels) != 3:\\n            raise ValueError(\\"Reader initialization requires a three-label NLI model\\")\\n        order = nli_label_order(reader.config.id2label, label_order)\\n        config = PairReaderConfig(encoder_config=reader.config.to_dict(), nli_labels=order,\\n                                  class_weights=class_weights, distillation_weight=distillation_weight,\\n                                  freeze_word_embeddings=freeze_word_embeddings, loss_normalization=loss_normalization)\\n        model = cls(config)\\n        model.reader = reader\\n        if freeze_word_embeddings:\\n            model.reader.get_input_embeddings().requires_grad_(False)\\n        return model\\n\\n    def forward(self, input_ids, attention_mask, evidence_mask, quality, teacher_probabilities,\\n                token_type_ids=None, labels=None):\\n        batch, passages, length = input_ids.shape\\n        encoded = {\\"input_ids\\": input_ids.reshape(-1, length), \\"attention_mask\\": attention_mask.reshape(-1, length)}\\n        if token_type_ids is not None:\\n            encoded[\\"token_type_ids\\"] = token_type_ids.reshape(-1, length)\\n        output = self.reader(**encoded, output_hidden_states=True, return_dict=True)\\n        hidden = output.hidden_states[-1][:, 0].reshape(batch, passages, -1)\\n        nli = output.logits.float().reshape(batch, passages, 3)\\n        indices = [self.config.nli_labels.index(k) for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")]\\n        nli = nli[..., indices]\\n        probabilities = nli.softmax(-1)\\n        valid = evidence_mask.to(hidden.dtype)\\n        projected = self.project(torch.cat([hidden, probabilities.to(hidden.dtype), quality.to(hidden.dtype)], dim=-1))\\n        attention_logits = self.attention(projected).squeeze(-1).masked_fill(valid == 0, -1e4)\\n        weights = attention_logits.softmax(-1) * valid\\n        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-6)\\n        pooled = (projected * weights.unsqueeze(-1)).sum(1)\\n        any_evidence = (valid.sum(1, keepdim=True) > 0).to(hidden.dtype)\\n        maximum = projected.masked_fill(valid.unsqueeze(-1) == 0, -1e4).max(1).values * any_evidence\\n        mean_nli = (probabilities * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)\\n        max_nli = probabilities.masked_fill(valid.unsqueeze(-1) == 0, 0).max(1).values\\n        mean_quality = (quality * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)\\n        conflict = max_nli[:, :1] * max_nli[:, 1:2]\\n        features = torch.cat([pooled, maximum, mean_nli, max_nli, mean_quality, any_evidence, conflict], dim=-1)\\n        logits = self.verdict(features.to(hidden.dtype))\\n        loss = None\\n        if labels is not None:\\n            loss = weighted_verdict_loss(logits, labels, self.verdict_weights, self.config.loss_normalization)\\n            if self.config.distillation_weight:\\n                teacher = teacher_probabilities.float().clamp_min(1e-7)\\n                kl = nn.functional.kl_div(nli.log_softmax(-1), teacher, reduction=\\"none\\").sum(-1)\\n                loss = loss + self.config.distillation_weight * (kl * valid).sum() / valid.sum().clamp_min(1)\\n        return SequenceClassifierOutput(loss=loss, logits=logits)\\n\\n\\nclass PairDataset(torch.utils.data.Dataset):\\n    def __init__(self, records, tokenizer, *, max_length=384, claim_budget=96, show_progress=True):\\n        from src.utils.progress import progress\\n\\n        self.records, self.features = records, []\\n        with progress(total=len(records), description=\\"Encode evidence pairs\\", unit=\\"claim\\", leave=False, disable=not show_progress) as bar:\\n            for record in records:\\n                hypothesis = tokenizer.decode(tokenizer.encode(record[\\"claim\\"], add_special_tokens=False)[:claim_budget], skip_special_tokens=True)\\n                evidence = (record.get(\\"retrieved_evidence\\") or [])[:3]\\n                pairs, mask, quality, teacher = [], [], [], []\\n                for i in range(3):\\n                    item = evidence[i] if i < len(evidence) else {\\"text\\": \\"No evidence available.\\"}\\n                    pairs.append(tokenizer(pair_premise(item), hypothesis, truncation=\\"only_first\\", max_length=max_length))\\n                    mask.append(float(i < len(evidence)))\\n                    quality.append(item.get(\\"quality\\", [0.0] * len(QUALITY_NAMES)))\\n                    p = item.get(\\"nli_probabilities\\", {\\"entailment\\": 0.0, \\"contradiction\\": 0.0, \\"neutral\\": 1.0})\\n                    teacher.append([p[k] for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")])\\n                feature = {\\"pairs\\": pairs, \\"evidence_mask\\": mask, \\"quality\\": quality,\\n                           \\"teacher_probabilities\\": teacher}\\n                if record.get(\'label\') is not None:\\n                    feature[\'labels\'] = CANONICAL_LABELS.index(record[\'label\'])\\n                self.features.append(feature)\\n                bar.update(1)\\n\\n    def __len__(self):\\n        return len(self.features)\\n\\n    def __getitem__(self, i):\\n        return self.features[i]\\n\\n\\nclass PairCollator:\\n    def __init__(self, tokenizer):\\n        self.tokenizer = tokenizer\\n\\n    def __call__(self, rows):\\n        padded = self.tokenizer.pad([p for r in rows for p in r[\\"pairs\\"]], return_tensors=\\"pt\\", pad_to_multiple_of=8)\\n        result = {k: v.reshape(len(rows), 3, -1) for k, v in padded.items()}\\n        for key in (\\"evidence_mask\\", \\"quality\\", \\"teacher_probabilities\\"):\\n            result[key] = torch.tensor([r[key] for r in rows], dtype=torch.float32)\\n        labeled = [\'labels\' in row for row in rows]\\n        if any(labeled) and not all(labeled):\\n            raise ValueError(\'Do not mix labeled training rows and unlabeled inference rows\')\\n        if all(labeled):\\n            result[\\"labels\\"] = torch.tensor([r[\\"labels\\"] for r in rows], dtype=torch.long)\\n        return result\\n"}')
SOURCE_SHA256 = 'b46efdb1896b46c1ba817de39a57ca3da6a4a085017365c6dcaa18dd1c51611b'
assert hashlib.sha256(json.dumps(SOURCES, sort_keys=True, ensure_ascii=True).encode()).hexdigest() == SOURCE_SHA256
RUNTIME = Path('/content') / ('claim_check_source_' + SOURCE_SHA256[:12])
for name, module in list(sys.modules.items()):
    if name.startswith('claim_check_colab_runtime_') and name != 'claim_check_colab_runtime_' + SOURCE_SHA256[:12] and getattr(module, '_ACTIVE', None):
        raise RuntimeError('A different backend source is still loaded. Use a fresh CPU session or stop the previous backend first.')
    if name == 'src' and getattr(module, '__file__', None) and not Path(module.__file__).is_relative_to(RUNTIME):
        raise RuntimeError('Different project code is already imported. Restart the kernel before running this notebook.')
for name, content in SOURCES.items():
    path = RUNTIME / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
sys.path.insert(0, str(RUNTIME))
sys.path.insert(0, str(RUNTIME / 'apps/backend'))
print('Portable source:', SOURCE_SHA256)


## 2. Install the text-only model dependencies

In [ ]:
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '4.57.1':
    raise RuntimeError('Use a fresh session before loading this notebook.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(RUNTIME / 'apps/backend/requirements-model.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| inference device: CPU')


## 3. Mount Drive and locate the serving inputs

Keep the shared shortcut named `10718_averitec`. A CLI-mounted Drive is reused. Only the model, tokenizer, run settings and saved dev rankings are read; no training labels or full passage corpus are needed.

In [ ]:
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
CACHE_ROOT = Path('/content/claim_check_serving')
PORT = 8000
CPU_THREADS = 2
if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Check shared-folder access and the My Drive shortcut: ' + str(DATA_ROOT))
print('Shared input root:', DATA_ROOT)


## 4. Copy the small serving bundle and start the private backend

Complete cached files are reused after interruption. Shared Drive files are never changed. This returns to the kernel after starting a background HTTP thread; there is no blocking server loop or artificial keepalive. Rerunning with the same source/settings reuses the loaded backend.

In [ ]:
module_name = 'claim_check_colab_runtime_' + SOURCE_SHA256[:12]
if module_name not in sys.modules:
    spec = importlib.util.spec_from_file_location(module_name, RUNTIME / 'apps/backend/colab_runtime.py')
    backend = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = backend
    spec.loader.exec_module(backend)
else:
    backend = sys.modules[module_name]
BACKEND_INFO = backend.start_backend(DATA_ROOT, CACHE_ROOT, port=PORT, cpu_threads=CPU_THREADS)
print(json.dumps(BACKEND_INFO, indent=2))


## 5. Check one real inference request

In [ ]:
import urllib.request
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
base = BACKEND_INFO['url']
with opener.open(base + '/health', timeout=10) as reply:
    health = json.load(reply)
assert health['adapter'] == 'model_registry', health
with opener.open(base + '/models', timeout=10) as reply:
    models = json.load(reply)
print('Available models:', [m['label'] for m in models['models'] if m['available']])
with opener.open(base + '/benchmark/claims?limit=1', timeout=10) as reply:
    claim_id = json.load(reply)['claims'][0]['claim_id']
request = urllib.request.Request(base + '/verify',
    data=json.dumps({'mode': 'benchmark', 'claim_id': claim_id, 'model_id': models['default_model_id']}).encode(),
    headers={'Content-Type': 'application/json'})
started = time.monotonic()
with opener.open(request, timeout=90) as reply:
    result = json.load(reply)
assert result['status'] in ('completed', 'insufficient_sources'), result
print('Claim:', claim_id, '| status:', result['status'], '| verdict:', result.get('verdict'))
print('Evidence items:', len(result['evidence']), '| runtime inference seconds:', round(time.monotonic() - started, 2))
print('Backend is ready for the local relay. No gold verdict labels were read.')


## 6. Optional: package this matching extension version

The zip contains extension code only, not Drive files, weights, credentials or predictions. Download with the Colab file browser, extract locally, then use Chrome's Load unpacked on the extracted folder.

In [ ]:
import zipfile
extension_zip = Path('/content/claim_check_extension.zip')
with zipfile.ZipFile(extension_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((RUNTIME / 'apps/browser-extension').iterdir()):
        if path.is_file():
            archive.write(path, path.name)
print('Extension zip:', extension_zip)
print('Source snapshot:', SOURCE_SHA256)


## Connect and stop

On your computer, run the repository's `scripts/colab_extension.ps1 -Action Relay`
or the Python relay command in `apps/COLAB_BACKEND.md`. Keep that terminal open.
Load the unpacked extension, use **Benchmark**, select **Model / evidence pipeline**, and keep Backend URL at
`http://127.0.0.1:8000`. Mock mode does not call the trained model; Live is unavailable.

Run `colab sessions` if you opened this notebook in the browser; select that same
session in the relay, rather than creating another one. CLI command startup adds
transport latency to the first verification of each claim; repeated inference is
cached separately for each pipeline in the remote adapter. Record this overhead if measuring decision time.

If the kernel restarts, rerun the notebook in order. A replacement VM needs a new
Drive mount and cache copy. Changing settings while the backend is loaded requires
`backend.stop_backend()` first. A new notebook source needs a fresh runtime.

When finished, stop the relay with Ctrl+C and stop the named Colab runtime:
`colab stop -s claim-check-extension`. Closing the extension does not stop the VM.